In [6]:
##Code description: After encapsulation testing, this code serves as the final experimental and released product. 
##It retains the Smart template method as the baseline approach for extreme testing scenarios (such as no text or network information input).
import pandas as pd
import networkx as nx
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import Counter, defaultdict
import re
import random
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import glob
import os
import json
import time
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Any, Set
import langdetect
from langdetect import detect, DetectorFactory
# Add at the beginning of imports:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

# Ensure reproducibility of language detection
DetectorFactory.seed = 0

# Import TextBlob for sentiment analysis
try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
    print("TextBlob successfully imported")
except ImportError:
    TEXTBLOB_AVAILABLE = False
    print("TextBlob not installed, using dictionary-based sentiment analysis")

# ========== Time-Aware Random Walk Particle Class ==========

class TimeAwareParticle:
    """Time-aware random competitive learning particle for detecting recent network environment"""
    
    def __init__(self, particle_id: int, start_node: str, network: nx.Graph, 
                 user_data: Dict[str, List[Tuple[str, datetime]]], 
                 max_steps: int = 8, time_decay_factor: float = 0.1):
        """
        Initialize time-aware particle
        
        Args:
            particle_id: Particle ID
            start_node: Starting node (user)
            network: Social network graph
            user_data: User data dictionary {username: [(post content, timestamp), ...]}
            max_steps: Maximum walk steps
            time_decay_factor: Time decay factor (larger value gives higher weight to recent posts)
        """
        self.particle_id = particle_id
        self.current_node = start_node
        self.network = network
        self.user_data = user_data
        self.max_steps = max_steps
        self.time_decay_factor = time_decay_factor
        self.visited_nodes = [start_node]
        self.collected_data = []  # Collected (post content, timestamp) data
        self.collected_posts = []  # Only post content
        self.step_count = 0
        self.current_time = self._get_current_time()  # Get current time (latest time in dataset)
        
        # Collect data from starting node
        self._collect_node_data(start_node)
    
    def _get_current_time(self) -> datetime:
        """Get the latest time in the dataset (as current time)"""
        all_times = []
        for posts_with_time in self.user_data.values():
            for _, post_time in posts_with_time:
                if post_time:
                    all_times.append(post_time)
        
        if all_times:
            return max(all_times)
        else:
            return datetime.now()
    
    def _get_time_weight(self, post_time: datetime) -> float:
        """Calculate time weight: more recent posts get higher weight"""
        if not post_time:
            return 0.1
        
        # Calculate time difference from current time (hours)
        time_diff_hours = (self.current_time - post_time).total_seconds() / 3600
        
        # Exponential decay function: weight = exp(-time_decay_factor * time_diff)
        weight = np.exp(-self.time_decay_factor * time_diff_hours)
        
        # Ensure weight is within reasonable range
        return max(0.1, min(1.0, weight))
    
    def _collect_node_data(self, node: str):
        """Collect node data, considering time weights"""
        if node not in self.user_data:
            return
        
        posts_with_time = self.user_data[node]
        
      
        weighted_posts = []
        for post_content, post_time in posts_with_time[:5]:  # Upper limit of posts per node to read
            weight = self._get_time_weight(post_time)
            weighted_posts.append((post_content, post_time, weight))
        
        # Sort by weight in descending order
        weighted_posts.sort(key=lambda x: x[2], reverse=True)
        
        # Take the highest weighted content
        for post_content, post_time, _ in weighted_posts[:3]:
            self.collected_data.append((post_content, post_time))
            self.collected_posts.append(post_content)
    
    def step(self) -> bool:
        """Perform one walk step, return whether to continue walking"""
        if self.step_count >= self.max_steps:
            return False
        
        # Get neighbors of current node
        neighbors = list(self.network.neighbors(self.current_node))
        
        if not neighbors:
            return False
        
        # Select next node based on time and content similarity
        next_node = self._time_aware_selection(neighbors)
        
        # Move to next node
        self.current_node = next_node
        self.visited_nodes.append(next_node)
        self._collect_node_data(next_node)
        self.step_count += 1
        
        return self.step_count < self.max_steps
    
    def _time_aware_selection(self, neighbors: List[str]) -> str:
        """Time-aware node selection"""
        selection_scores = []
        
        for neighbor in neighbors:
            # Base similarity score
            base_score = 0.1
            
            if self.network.has_edge(self.current_node, neighbor):
                base_score = self.network[self.current_node][neighbor].get('weight', 0.1)
            
            # Time score: if neighbor has recent posts, increase probability of being selected
            time_score = 0.0
            if neighbor in self.user_data and self.user_data[neighbor]:
                # Get latest post time of neighbor
                latest_time = None
                for _, post_time in self.user_data[neighbor]:
                    if post_time:
                        if latest_time is None or post_time > latest_time:
                            latest_time = post_time
                
                if latest_time:
                    # Calculate time weight
                    time_score = self._get_time_weight(latest_time) * 0.3
            
            # Avoid repeated visits
            if neighbor in self.visited_nodes:
                penalty = -0.2
            else:
                penalty = 0.0
            
            # Combined score = base similarity + time score + random exploration + repeat visit penalty
            total_score = base_score + time_score + random.uniform(0, 0.05) + penalty
            selection_scores.append(max(0.01, total_score))
        
        # Probability selection based on scores
        total_score = sum(selection_scores)
        if total_score > 0:
            probabilities = [score / total_score for score in selection_scores]
            return random.choices(neighbors, weights=probabilities, k=1)[0]
        else:
            return random.choice(neighbors)
    
    def get_collected_posts(self) -> List[str]:
        """Get collected post content"""
        return self.collected_posts
    
    def get_collected_data_with_time(self) -> List[Tuple[str, datetime]]:
        """Get collected data (including timestamps)"""
        return self.collected_data
    
    def get_visited_nodes(self) -> List[str]:
        """Get visited nodes"""
        return self.visited_nodes

# ========== Smart Large Model Wrapper ==========

class SmartModelWrapper:
    """Smart large model wrapper - learns user style"""
    
    def __init__(self, model_path="C:/Users/fs/Desktop/lwz/models/MiniCPM-4.0-8B"):
        self.model_path = model_path
        self.tokenizer = None
        self.model = None
        self.device = None
        self._load_model()
    

    def _load_model(self):
        try:
            print(f"Loading model: {self.model_path}")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_path, trust_remote_code=True)
    
            if torch.cuda.is_available():
                self.device = torch.device("cuda:0")
                print(f"Using GPU: {torch.cuda.get_device_name(0)} Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
                
                # Explicitly load to cuda:0, using float16
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_path,
                    torch_dtype=torch.float16,
                    device_map={"": 0},   # Assign all layers to GPU 0
                    trust_remote_code=True,
                    low_cpu_mem_usage=True
                )
            else:
                self.device = torch.device("cpu")
                print("Using CPU")
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_path,
                    torch_dtype=torch.float32,
                    device_map=None,
                    trust_remote_code=True
                ).to(self.device)
            
            print("Model loaded successfully")
            # Verify device
            print(f"Model device: {next(self.model.parameters()).device}")
    
        except Exception as e:
            print(f"Model loading failed: {e}")
            self._load_model_fallback()

         
    
    def _load_model_fallback(self):
        """Model loading fallback"""
        print("Attempting fallback model loading...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_path, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_path,
                torch_dtype=torch.float32,
                device_map=None,
                trust_remote_code=True
            ).to('cpu')
            self.device = torch.device('cpu')
            print("Model loaded successfully (fallback)")
        except Exception as e:
            print(f"Fallback also failed: {e}")
            raise
    
    def analyze_user_style(self, user_posts: List[str]) -> Dict:
        """Deeply analyze user style"""
        if not user_posts:
            return {
                'language': 'en',
                'style': 'neutral',
                'common_words': [],
                'sentence_patterns': [],
                'emotional_tone': 'neutral',
                'sample_phrases': []
            }
        
        # Analyze language
        language = self._detect_language(user_posts)
        
        # Analyze common words
        common_words = self._extract_common_words(user_posts, language)
        
        # Analyze sentence patterns
        sentence_patterns = self._analyze_sentence_patterns(user_posts)
        
        # Analyze emotional tone
        emotional_tone = self._analyze_emotional_tone(user_posts)
        
        # Extract sample phrases
        sample_phrases = self._extract_sample_phrases(user_posts)
        
        # Build style description
        style_desc = self._build_style_description(language, sentence_patterns, emotional_tone)
        
        return {
            'language': language,
            'style': style_desc,
            'common_words': common_words[:10],
            'sentence_patterns': sentence_patterns,
            'emotional_tone': emotional_tone,
            'sample_phrases': sample_phrases[:5],
            'post_samples': user_posts[:3]
        }
    
    def _detect_language(self, posts: List[str]) -> str:
        """Detect user's primary language"""
        if not posts:
            return 'en'
        
        lang_counts = {}
        for post in posts[:10]:
            if isinstance(post, str) and len(post.strip()) > 10:
                try:
                    lang = detect(post)
                    lang_counts[lang] = lang_counts.get(lang, 0) + 1
                except:
                    continue
        
        if lang_counts:
            return max(lang_counts.items(), key=lambda x: x[1])[0]
        return 'en'
    
    def _extract_common_words(self, posts: List[str], language: str) -> List[str]:
        """Extract user's common words"""
        if not posts:
            return []
        
        all_text = ' '.join(posts).lower()
        
        # Choose segmentation method based on language
        if language in ['ja', 'zh-cn', 'zh-tw', 'ko']:
            # For East Asian languages, extract character sequences
            words = re.findall(r'[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff\uac00-\ud7af]{2,}', all_text)
        else:
            # For other languages, extract words
            words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text)
        
        # Filter stopwords
        if language == 'en':
            stopwords = {'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with', 'have', 'from'}
            words = [w for w in words if w not in stopwords]
        elif language == 'ja':
            stopwords = {'です', 'ます', 'は', 'の', 'に', 'を', 'が', 'で', 'と', 'も'}
            words = [w for w in words if w not in stopwords]
        
        word_freq = Counter(words)
        return [word for word, _ in word_freq.most_common(20)]
    
    def _analyze_sentence_patterns(self, posts: List[str]) -> List[str]:
        """Analyze user's sentence patterns"""
        patterns = []
        
        for post in posts[:10]:
            if isinstance(post, str):
                # Analyze sentence structure
                if '?' in post:
                    patterns.append('questioning')
                if '!' in post:
                    patterns.append('exclamatory')
                if '...' in post or '…' in post:
                    patterns.append('pausing')
                if len(post.split()) < 10:
                    patterns.append('concise')
                elif len(post.split()) > 25:
                    patterns.append('elaborate')
        
        # Deduplicate and take most common patterns
        pattern_counts = Counter(patterns)
        return [pattern for pattern, _ in pattern_counts.most_common(3)]
    
    def _analyze_emotional_tone(self, posts: List[str]) -> str:
        """Analyze user's emotional tone"""
        if not posts:
            return 'neutral'
        
        positive_words = ['good', 'great', 'excellent', 'positive', 'optimistic', 'happy', 'love', 'like']
        negative_words = ['bad', 'poor', 'negative', 'problem', 'issue', 'hate', 'dislike']
        
        pos_count = 0
        neg_count = 0
        
        for post in posts[:10]:
            if isinstance(post, str):
                text_lower = post.lower()
                pos_count += sum(1 for word in positive_words if word in text_lower)
                neg_count += sum(1 for word in negative_words if word in text_lower)
        
        if pos_count > neg_count * 2:
            return 'positive'
        elif neg_count > pos_count * 2:
            return 'negative'
        else:
            return 'neutral'
    
    def _extract_sample_phrases(self, posts: List[str]) -> List[str]:
        """Extract user's characteristic phrases"""
        phrases = []
        
        for post in posts[:10]:
            if isinstance(post, str):
                # Extract short sentences (5-15 words)
                sentences = re.split(r'[.!?]+', post)
                for sentence in sentences:
                    words = sentence.strip().split()
                    if 5 <= len(words) <= 15:
                        phrases.append(sentence.strip())
        
        return phrases[:10]
    
    def _build_style_description(self, language: str, patterns: List[str], tone: str) -> str:
        """Build style description"""
        style_parts = []
        
        # Add language information
        lang_map = {'en': 'English', 'ja': 'Japanese', 'zh-cn': 'Chinese', 'ko': 'Korean'}
        lang_desc = lang_map.get(language, language)
        style_parts.append(f"writes in {lang_desc}")
        
        # Add sentence patterns
        if patterns:
            pattern_desc = ', '.join(patterns[:2])
            style_parts.append(f"uses {pattern_desc} sentences")
        
        # Add emotional tone
        if tone == 'positive':
            style_parts.append("generally positive tone")
        elif tone == 'negative':
            style_parts.append("often critical tone")
        
        return ', '.join(style_parts)
    
    def generate_context_aware_post(self, user_style: Dict, network_context: Dict, 
                                   positivity_boost: float = 0.5) -> str:
        """Generate environment-aware post - fully based on large model, no templates"""
        if self.model is None or self.tokenizer is None:
            raise ValueError("Large model not loaded, cannot generate content")
        
        # Build intelligent prompt
        prompt = self._build_intelligent_prompt(user_style, network_context, positivity_boost)
        
        try:
            # Encode input
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            
            # Get the device where the model is located
            model_device = None
            if hasattr(self.model, 'device'):
                model_device = self.model.device
            else:
                try:
                    first_param = next(self.model.parameters())
                    model_device = first_param.device
                except:
                    # If unable to get, use initialized device
                    model_device = self.device
            
            print(f"  Model device: {model_device}")
            
            # Ensure input data is on the correct device
            inputs = {k: v.to(model_device) for k, v in inputs.items()}
            
            # Generation configuration
            generation_config = {
                'max_new_tokens': 128,  # Reduce generation length for speed
                'temperature': 0.8,
                'do_sample': True,
                'top_p': 0.9,
                'repetition_penalty': 1.1,
                'pad_token_id': self.tokenizer.eos_token_id,
                'num_return_sequences': 1
            }
            
            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    **generation_config
                )
            
            # Decode output
            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Extract response
            result = self._extract_response(generated_text, prompt)
            print(f"  Generation successful: {result[:50]}...")
            
            return result
            
        except Exception as e:
            print(f"Large model generation failed: {e}")
            import traceback
            traceback.print_exc()
            # Fallback: return simple generation
            language = user_style.get('language', 'en')
            if language == 'zh-cn':
                return "This discussion on this topic is quite interesting."
            elif language == 'ja':
                return "この話題についての議論は興味深いです。"
            else:
                return "Interesting discussion on this topic."
    
    def _build_intelligent_prompt(self, user_style: Dict, network_context: Dict, 
                                 positivity_boost: float) -> str:
        """Build intelligent prompt"""
        # User style part
        user_desc = f"""User Writing Style Analysis:
- Primary language: {user_style.get('language', 'en')}
- Style characteristics: {user_style.get('style', 'neutral')}
- Common words/phrases: {', '.join(user_style.get('common_words', [])[:5])}
- Sentence patterns: {', '.join(user_style.get('sentence_patterns', []))}
- Emotional tone: {user_style.get('emotional_tone', 'neutral')}

Recent user posts:
{chr(10).join(['- ' + post for post in user_style.get('post_samples', [])[:2]])}"""
        
        # Network environment part
        env_desc = "Current Network Environment:\n"
        
        if network_context.get('sentiment'):
            env_desc += f"- Overall sentiment: {network_context['sentiment']}\n"
        
        if network_context.get('trending_words'):
            trending = network_context['trending_words'][:5]
            env_desc += f"- Trending words: {', '.join(trending)}\n"
        
        if network_context.get('recent_topics'):
            topics = network_context['recent_topics'][:3]
            env_desc += f"- Recent topics: {', '.join(topics)}\n"
        
        if network_context.get('consensus_analysis'):
            env_desc += f"- Network consensus: {network_context['consensus_analysis']}\n"
        
        if network_context.get('time_frame'):
            env_desc += f"- Time frame: {network_context['time_frame']}\n"

        # At the end of env_desc building, add neighbor post information if available
        if network_context.get('neighbor_posts'):
            neighbor_posts = network_context['neighbor_posts'][:3]  # Maximum 3 posts
            env_desc += "- Neighbor posts for reference:\n"
            for p in neighbor_posts:
                env_desc += f"  - {p}\n"
        
        # Generation strategy
        strategy = "Generation Strategy:\n"
        
        # Adjust strategy based on network consensus
        consensus = network_context.get('consensus_analysis', '')
        if "positive consensus" in consensus:
            strategy += "- Network has positive consensus, you can be more direct\n"
        elif "negative consensus" in consensus:
            strategy += "- Network has negative consensus, express views diplomatically\n"
        
        # Adjust strategy based on network sentiment
        sentiment = network_context.get('sentiment', 'neutral')
        if sentiment in ["generally positive", "slightly positive"]:
            strategy += "- Overall sentiment is positive, match this tone\n"
        elif sentiment in ["generally negative", "slightly negative"]:
            strategy += "- Overall sentiment is negative, be cautious but constructive\n"
        
        # Adjust based on positivity
        if positivity_boost > 0.7:
            strategy += "- Use strong positive language, show enthusiasm\n"
        elif positivity_boost > 0.4:
            strategy += "- Use moderately positive language\n"
        else:
            strategy += "- Keep tone balanced, slightly positive\n"
        
        # Assemble prompt
        prompt = f"""You are an AI assistant tasked with simulating social media posts. Your goal is to generate a post that perfectly matches a user's style while responding to the current network environment.

{user_desc}

{env_desc}

{strategy}

CRITICAL REQUIREMENTS:
1. Write in the EXACT same language as the user's recent posts
2. Match the user's writing style, vocabulary, and sentence patterns
3. Reference the network environment appropriately (use trending words/topics)
4. Adjust tone based on network sentiment and consensus
5. Sound authentic, like the user wrote it themselves
6. Length: 1-2 sentences (matching user's typical length)

Generate the post exactly as the user would write it:"""
        
        return prompt
    
    def _extract_response(self, generated_text: str, prompt: str) -> str:
        """Extract response from generated text"""
        if generated_text.startswith(prompt):
            response = generated_text[len(prompt):].strip()
        else:
            # Try to find generated content
            markers = ["Generate the post:", "Generated post:", "Post:", "内容:", "生成内容:"]
            for marker in markers:
                idx = generated_text.find(marker)
                if idx != -1:
                    response = generated_text[idx + len(marker):].strip()
                    break
            else:
                # Use last part
                lines = generated_text.split('\n')
                response = lines[-1].strip() if lines else generated_text
        
        # Clean response
        response = re.sub(r'\s+', ' ', response).strip()
        
        # Remove possible quotes or explanations
        response = re.sub(r'^"(.*)"$', r'\1', response)
        response = re.sub(r'^(As|So|Well|Actually|Basically),?\s*', '', response, flags=re.IGNORECASE)
        
        return response


# ============================================================================
# TWICEMethod - Simplified Author-Implemented Reproduction of TWICE
# ============================================================================
# Reference: Jin et al., "TWICE: An LLM Agent Framework for Simulating
#            Personalized User Tweeting Behavior with Long-term Temporal Features"
#            (arXiv:2602.22222, 2025)
#
# Core ideas preserved from the original TWICE:
#   1. Personalized user style modeling based on historical posts
#   2. Social neighborhood context integration for generation
#   3. LLM-based user simulation for social media posting
#
# Engineering simplifications (baseline-only, not the focus of this work):
#   - TF-IDF cosine similarity is used for neighbor retrieval instead of
#     the full event-driven memory module with temporal decay
#   - Single-stage generation is used instead of the two-stage
#     content-then-style rewriting workflow
#   - No long-term temporal event modeling (time window / state coefficient)
#
# Note: The original TWICE had no official open-source release at the time
#       of our experiments. This implementation serves as a modern baseline
#       for architectural comparison with our proposed SA-SCL framework.
# ============================================================================
class TWICEMethod:
    """AI large model-based social network cloning generation model TWICE baseline method
       Uses user's own style + posts from most similar neighbors as context
    """
    def __init__(self, train_df, model_wrapper, top_k_neighbors=3):
        self.train_df = train_df
        self.model = model_wrapper
        self.top_k = top_k_neighbors
        self.user_posts = defaultdict(list)
        self.user_list = []
        self.vectors = None
        self.vectorizer = None
        self._build_user_profiles()

    def _build_user_profiles(self):
        """Build each user's post collection and TF-IDF vectors for similarity calculation"""
        for _, row in self.train_df.iterrows():
            if pd.notna(row['text']):
                self.user_posts[row['userName']].append(str(row['text']))
        self.user_list = list(self.user_posts.keys())
        if len(self.user_list) < 2:
            return  # Too few users to calculate neighbors
        texts = [' '.join(posts) for posts in self.user_posts.values()]
        self.vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
        self.vectors = self.vectorizer.fit_transform(texts)

    def find_neighbors(self, user_name):
        """Find the top_k most similar other users to the target user"""
        if user_name not in self.user_posts or self.vectors is None:
            return []
        idx = self.user_list.index(user_name)
        vec = self.vectors[idx]
        similarities = cosine_similarity(vec, self.vectors).flatten()
        similar_indices = np.argsort(similarities)[::-1]
        neighbors = []
        for i in similar_indices:
            if self.user_list[i] != user_name:
                neighbors.append(self.user_list[i])
                if len(neighbors) >= self.top_k:
                    break
        return neighbors

    def generate_for_user(self, user_name, num_posts=1):
        """Generate one post: combine user style and recent neighbor posts"""
        user_posts = self.user_posts.get(user_name, [])
        if not user_posts:
            return [f"TWICE: User {user_name} has no historical posts"]

        # 1. Analyze user style
        user_style = self.model.analyze_user_style(user_posts)

        # 2. Get neighbor posts as environmental context
        neighbors = self.find_neighbors(user_name)
        neighbor_posts = []
        for neigh in neighbors[:self.top_k]:
            posts = self.user_posts.get(neigh, [])
            if posts:
                # Take the most recent 2 (assuming data order is roughly chronological, otherwise could sort further)
                neighbor_posts.extend(posts[:2])

        # 3. Build network context (add neighbor_posts key)
        network_context = {
            'sentiment': 'neutral',
            'trending_words': [],
            'recent_topics': [],
            'consensus_analysis': 'mixed opinions',
            'time_frame': 'recent',
            'neighbor_posts': neighbor_posts[:5]  # Maximum 5 posts
        }

        # 4. Call large model to generate
        try:
            post = self.model.generate_context_aware_post(user_style, network_context, positivity_boost=0.5)
            return [post]
        except Exception as e:
            print(f"  TWICE generation failed: {e}")
            # Fallback: generate based only on user style
            fallback_context = {'sentiment': 'neutral', 'trending_words': [], 'recent_topics': []}
            post = self.model.generate_context_aware_post(user_style, fallback_context, 0.5)
            return [post]


            
class IntelligentSCLMethod:
    """Intelligent SCL method - deep user style learning and environment integration"""
    
    def __init__(self, train_df, model_wrapper, positivity_boost: float = 0.5, 
                 n_particles: int = 5, max_steps_per_particle: int = 6,
                 recent_hours: int = 48, time_decay_factor: float = 0.1):
        """
        Initialize intelligent SCL method
        
        Args:
            train_df: Training data
            model_wrapper: Large model wrapper
            positivity_boost: Positivity boost coefficient
            n_particles: Number of particles
            max_steps_per_particle: Maximum steps per particle
            recent_hours: Time window for defining "recent" (hours)
            time_decay_factor: Time decay factor
        """
        self.train_df = train_df
        self.model = model_wrapper
        self.positivity_boost = positivity_boost
        self.n_particles = n_particles
        self.max_steps_per_particle = max_steps_per_particle
        self.recent_hours = recent_hours
        self.time_decay_factor = time_decay_factor
        
        # Parse timestamps
        self.train_df = self._parse_timestamps(train_df)
        
        # Check if data is sufficient
        if len(train_df) < 3:
            raise ValueError(f"Insufficient user data (only {len(train_df)} posts), cannot perform analysis")
        
        # Get recently active users and data
        self.recent_data = self._get_recent_data(recent_hours)
        
        # Build recent social network
        self.network = self._build_recent_social_network()
        
        # Prepare recent user data (including timestamps)
        self.user_data_with_time = self._prepare_user_data_with_time()
        
        # Use time-aware particles to explore recent network environment
        self.network_context = self._explore_recent_network_environment()
        
        # Extract user profiles (using all historical data)
        self.user_profiles = self._extract_user_profiles()
        
        print(f"  Intelligent SCL method initialization complete:")
        print(f"    - Time window: last {recent_hours} hours")
        print(f"    - Recent post count: {len(self.recent_data)}")
        print(f"    - Active user count: {len(self.user_data_with_time)}")
        if self.network_context.get('trending_words'):
            print(f"    - Trending words: {self.network_context['trending_words'][:5]}")
    
    def _parse_timestamps(self, df):
        """Parse timestamp column"""
        df_copy = df.copy()
        
        # Check if timestamp column exists
        if 'timeStamp' in df_copy.columns:
            try:
                df_copy['parsed_time'] = pd.to_datetime(df_copy['timeStamp'], format='%Y-%m-%d %H:%M:%S')
                print(f"  Successfully parsed timestamp column, earliest: {df_copy['parsed_time'].min()}, latest: {df_copy['parsed_time'].max()}")
            except Exception as e:
                print(f"  Timestamp parsing failed: {e}")
                df_copy['parsed_time'] = pd.date_range(end=pd.Timestamp.now(), periods=len(df_copy), freq='h')
        else:
            df_copy['parsed_time'] = pd.date_range(end=pd.Timestamp.now(), periods=len(df_copy), freq='h')
        
        df_copy = df_copy.sort_values('parsed_time', ascending=True).reset_index(drop=True)
        return df_copy
    
    def _get_recent_data(self, recent_hours: int):
        """Get data from the last specified hours"""
        if 'parsed_time' not in self.train_df.columns:
            return self.train_df
        
        # Get latest time
        latest_time = self.train_df['parsed_time'].max()
        
        # Calculate time threshold
        time_threshold = latest_time - pd.Timedelta(hours=recent_hours)
        
        # Filter recent data
        recent_data = self.train_df[self.train_df['parsed_time'] >= time_threshold].copy()
        
        return recent_data
    
    def _build_recent_social_network(self) -> nx.Graph:
        """Build recent social network graph (based on recently active users)"""
        G = nx.Graph()
        
        # Get recently active users
        if len(self.recent_data) > 0:
            user_counts = self.recent_data['userName'].value_counts()
            recent_users = user_counts.index[:min(20, len(user_counts))]
        else:
            user_counts = self.train_df['userName'].value_counts()
            recent_users = user_counts.index[:min(20, len(user_counts))]
        
        print(f"  Building recent social network, active users: {len(recent_users)}")
        
        for user in recent_users:
            G.add_node(user)
        
        # Build edges based on content similarity
        for i, user1 in enumerate(recent_users):
            user1_posts = self.recent_data[self.recent_data['userName'] == user1]['text'].dropna().tolist()[:5]
            
            for user2 in recent_users[i+1:]:
                user2_posts = self.recent_data[self.recent_data['userName'] == user2]['text'].dropna().tolist()[:5]
                
                if user1_posts and user2_posts:
                    similarity = self._calculate_content_similarity(user1_posts, user2_posts)
                    
                    if similarity > 0.2:
                        G.add_edge(user1, user2, weight=similarity)
        
        print(f"  Network construction complete, nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
        return G
    
    def _calculate_content_similarity(self, posts1: List[str], posts2: List[str]) -> float:
        """Calculate content similarity"""
        if not posts1 or not posts2:
            return 0.0
        
        # Extract keywords
        def extract_keywords(posts):
            all_text = ' '.join([str(p) for p in posts])
            words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower()) + re.findall(r'[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff\uac00-\ud7af]{2,}', all_text)
            return set(words)
        
        keywords1 = extract_keywords(posts1)
        keywords2 = extract_keywords(posts2)
        
        if not keywords1 or not keywords2:
            return 0.0
        
        intersection = len(keywords1 & keywords2)
        union = len(keywords1 | keywords2)
        
        return intersection / union if union > 0 else 0.0
    
    def _prepare_user_data_with_time(self) -> Dict[str, List[Tuple[str, datetime]]]:
        """Prepare user data dictionary (including timestamps)"""
        user_data = {}
        
        # Get active users
        if len(self.recent_data) > 0:
            recent_users = self.recent_data['userName'].value_counts().index[:min(15, len(self.recent_data))]
        else:
            recent_users = self.train_df['userName'].value_counts().index[:min(15, len(self.train_df))]
        
        for user in recent_users:
            # Get recent posts of this user
            user_df = self.train_df[self.train_df['userName'] == user]
            user_df_sorted = user_df.sort_values('parsed_time', ascending=False)
            
            posts_with_time = []
            for _, row in user_df_sorted.iterrows():
                if pd.notna(row['text']):
                    posts_with_time.append((str(row['text']), row['parsed_time']))
            
            if posts_with_time:
                user_data[user] = posts_with_time[:8]
        
        return user_data
    
    def _explore_recent_network_environment(self) -> Dict:
        """Use time-aware particles to explore recent network environment"""
        print(f"  Using {self.n_particles} time-aware particles to explore recent network environment...")
        
        active_users = list(self.user_data_with_time.keys())
        if not active_users:
            print("  No active users, using default environment")
            return self._get_default_network_context()
        
        all_collected_posts = []
        
        for i in range(self.n_particles):
            start_user = random.choice(active_users)
            
            particle = TimeAwareParticle(
                particle_id=i,
                start_node=start_user,
                network=self.network,
                user_data=self.user_data_with_time,
                max_steps=self.max_steps_per_particle,
                time_decay_factor=self.time_decay_factor
            )
            
            # Perform walk
            for _ in range(self.max_steps_per_particle):
                if not particle.step():
                    break
            
            # Collect data
            collected_posts = particle.get_collected_posts()
            all_collected_posts.extend(collected_posts)
        
        # If collected too little data, supplement with some recent posts
        if len(all_collected_posts) < 10:
            recent_posts = self.recent_data['text'].dropna().tolist()[:20]
            all_collected_posts.extend(recent_posts)
        
        print(f"  Particle exploration complete, collected {len(all_collected_posts)} recent posts")
        
        # Analyze collected recent environment data
        return self._analyze_collected_data(all_collected_posts)
    
    def _analyze_collected_data(self, collected_posts: List[str]) -> Dict:
        """Analyze collected data"""
        if not collected_posts:
            return self._get_default_network_context()
        
        # Extract trending words
        trending_words = self._extract_trending_words(collected_posts)
        
        # Analyze sentiment
        sentiment = self._analyze_sentiment(collected_posts)
        
        # Extract topics
        recent_topics = self._extract_topics(collected_posts)
        
        # Analyze consensus
        consensus = self._analyze_consensus(collected_posts)
        
        return {
            'sentiment': sentiment,
            'trending_words': trending_words,
            'recent_topics': recent_topics,
            'consensus_analysis': consensus,
            'time_frame': f'last {self.recent_hours} hours',
            'data_source': 'particle_exploration'
        }
    
    def _extract_trending_words(self, posts: List[str]) -> List[str]:
        """Extract trending words"""
        if not posts:
            return []
        
        all_text = ' '.join([str(p) for p in posts[:30] if isinstance(p, str)])
        
        # Extract words from multiple languages
        words = []
        # English words
        words.extend(re.findall(r'\b[a-zA-Z]{4,}\b', all_text.lower()))
        # Chinese words
        words.extend(re.findall(r'[\u4e00-\u9fff]{2,}', all_text))
        # Japanese words
        words.extend(re.findall(r'[\u3040-\u309f\u30a0-\u30ff]{2,}', all_text))
        # Korean words
        words.extend(re.findall(r'[\uac00-\ud7af]{2,}', all_text))
        
        # Filter common stopwords
        stopwords = {
            'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with', 'have', 'from',
            'they', 'what', 'when', 'were', 'been', 'also', 'more', 'some', 'such',
            'only', 'just', 'like', 'will', 'than', 'other', 'there', 'their', 'them',
            'these', 'those', 'where', 'about', 'think', 'going', 'could', 'would',
            'should', 'which', 'while', 'after', 'before', 'during'
        }
        
        filtered_words = [w for w in words if w.lower() not in stopwords]
        
        # Calculate word frequency
        word_freq = Counter(filtered_words)
        
        # Return high frequency words
        return [word for word, freq in word_freq.most_common(15) if freq >= 2]
    
    def _analyze_sentiment(self, posts: List[str]) -> str:
        """Analyze sentiment"""
        if not posts:
            return "neutral"
        
        positive_count = 0
        negative_count = 0
        
        for post in posts[:20]:
            if isinstance(post, str):
                text_lower = post.lower()
                
                # Positive words
                pos_words = ['good', 'great', 'excellent', 'positive', 'optimistic', 
                           'happy', 'progress', 'success', 'improve', 'better',
                           'love', 'like', 'support', 'agree', 'encourage']
                
                # Negative words
                neg_words = ['bad', 'poor', 'negative', 'problem', 'issue', 'worry',
                           'concern', 'difficult', 'challenge', 'failure',
                           'hate', 'dislike', 'against', 'disagree', 'criticize']
                
                pos_count = sum(1 for word in pos_words if word in text_lower)
                neg_count = sum(1 for word in neg_words if word in text_lower)
                
                if pos_count > neg_count:
                    positive_count += 1
                elif neg_count > pos_count:
                    negative_count += 1
        
        total = positive_count + negative_count
        if total > 0:
            pos_ratio = positive_count / total
            
            if pos_ratio > 0.6:
                return "generally positive"
            elif pos_ratio > 0.4:
                return "slightly positive"
            elif pos_ratio < 0.2:
                return "generally negative"
            elif pos_ratio < 0.4:
                return "slightly negative"
        
        return "mixed/neutral"
    
    def _extract_topics(self, posts: List[str]) -> List[str]:
        """Extract topics"""
        if not posts:
            return []
        
        # Extract noun phrases (simple implementation)
        topics = []
        for post in posts[:15]:
            if isinstance(post, str):
                # Extract words starting with capital letters
                noun_phrases = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b', post)
                topics.extend([np for np in noun_phrases if len(np.split()) <= 3])
        
        topic_freq = Counter(topics)
        return [topic for topic, freq in topic_freq.most_common(10) if freq >= 2]
    
    def _analyze_consensus(self, posts: List[str]) -> str:
        """Analyze consensus"""
        if not posts:
            return "mixed opinions"
        
        sentiment = self._analyze_sentiment(posts)
        
        if "positive" in sentiment:
            return "moderate positive consensus"
        elif "negative" in sentiment:
            return "moderate negative consensus"
        else:
            return "mixed opinions"
    
    def _get_default_network_context(self) -> Dict:
        """Get default network environment"""
        return {
            'sentiment': 'neutral',
            'trending_words': [],
            'recent_topics': [],
            'consensus_analysis': 'mixed opinions',
            'time_frame': 'unknown',
            'data_source': 'default'
        }
    
    def _extract_user_profiles(self):
        """Extract user profiles"""
        profiles = {}
        
        user_counts = self.train_df['userName'].value_counts()
        users = user_counts.index[:min(10, len(user_counts))]
        
        for user in users:
            user_posts = self.train_df[self.train_df['userName'] == user]['text'].dropna().tolist()
            
            if not user_posts:
                continue
            
            # Use large model to analyze user style
            try:
                user_style = self.model.analyze_user_style(user_posts)
                profiles[user] = user_style
            except Exception as e:
                print(f"  Failed to analyze user {user} style: {e}")
                # Use simple analysis as fallback
                profiles[user] = self._simple_user_analysis(user_posts)
        
        return profiles
    
    def _simple_user_analysis(self, user_posts: List[str]) -> Dict:
        """User analysis (fallback)"""
        # Detect language
        language = 'en'
        try:
            sample_text = ' '.join(user_posts[:3])
            if len(sample_text) > 10:
                language = detect(sample_text)
        except:
            pass
        
        # Extract some keywords
        all_text = ' '.join(user_posts).lower()
        words = re.findall(r'\b[a-zA-Z]{4,}\b', all_text)
        word_freq = Counter(words)
        common_words = [word for word, _ in word_freq.most_common(10)]
        
        return {
            'language': language,
            'style': f'user writing in {language}',
            'common_words': common_words,
            'sentence_patterns': ['declarative'],
            'emotional_tone': 'neutral',
            'sample_phrases': user_posts[:2],
            'post_samples': user_posts[:3]
        }
    
    def generate_for_user(self, user_name: str, num_posts: int = 1) -> List[str]:
        """Generate content for user - based on large model learning"""
        # Check if user has historical data
        if user_name not in self.user_profiles:
            user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
            if not user_posts:
                raise ValueError(f"User {user_name} has no historical post data, cannot generate content")
            
            # If not in profiles but has data, analyze immediately
            user_style = self.model.analyze_user_style(user_posts)
        else:
            user_style = self.user_profiles[user_name]
        
        print(f"  [Intelligent Generation] User: {user_name}")
        print(f"  [Intelligent Generation] Detected language: {user_style.get('language', 'unknown')}")
        print(f"  [Intelligent Generation] User style: {user_style.get('style', 'unknown')}")
        if user_style.get('post_samples'):
            print(f"  [Intelligent Generation] User sample: {user_style['post_samples'][0][:50]}...")
        
        # Generate post
        try:
            post = self.model.generate_context_aware_post(
                user_style, 
                self.network_context, 
                self.positivity_boost
            )
            print(f"  [Intelligent Generation] Generated content: {post}")
            return [post]
            
        except Exception as e:
            print(f"  [Intelligent Generation] Generation failed: {e}")
            return self._generate_fallback_post(user_style, self.network_context)
    
    def _generate_fallback_post(self, user_style: Dict, network_context: Dict) -> List[str]:
        """Fallback generation - based on user style and network environment"""
        language = user_style.get('language', 'en')
        common_words = user_style.get('common_words', [])
        trending_words = network_context.get('trending_words', [])
        
        # Select topic words
        if trending_words:
            topic = random.choice(trending_words)
        elif common_words:
            topic = random.choice(common_words[:5])
        else:
            topic = "current developments"
        
        # Select generation template based on language
        if language == 'ja':  # Japanese
            templates = [
                f"{topic}について、興味深い議論が続いています。",
                f"{topic}の進展を見守っています。",
                f"{topic}に関して、良い議論がなされていると思います。",
            ]
        elif language == 'zh-cn':  # Chinese
            templates = [
                f"关于{topic}的讨论很有意义。",
                f"{topic}方面有新的进展。",
                f"{topic}这个话题值得关注。",
            ]
        elif language == 'ko':  # Korean
            templates = [
                f"{topic}에 대한 논의가 계속되고 있습니다.",
                f"{topic}의 발전을 지켜보고 있습니다.",
                f"{topic}에 대해 좋은 논의가 이루어지고 있습니다.",
            ]
        else:  # English or others
            templates = [
                f"Interesting discussions continuing about {topic}.",
                f"Following the developments on {topic}.",
                f"Good discussions happening around {topic}.",
            ]
        
        return [random.choice(templates)]

# ========== Other Methods ==========

class MiniCPMOnlyMethod:
    """Only use large model (without environment awareness)"""
    
    def __init__(self, train_df, model_wrapper):
        self.train_df = train_df
        self.model = model_wrapper
    
    def generate_for_user(self, user_name, num_posts=1):
        """Generate based only on user history"""
        user_posts = self.train_df[
            (self.train_df['userName'] == user_name) & 
            (self.train_df['text'].notna())
        ]['text'].tolist()
        
        if not user_posts:
            return [f"Generated content for {user_name}"]
        
        try:
            # Analyze user style
            user_style = self.model.analyze_user_style(user_posts)
            
            # Simple context
            context = {
                'sentiment': 'neutral',
                'trending_words': [],
                'recent_topics': [],
                'consensus_analysis': 'mixed opinions',
                'time_frame': 'recent'
            }
            
            # Generate post
            post = self.model.generate_context_aware_post(user_style, context, 0.5)
            return [post]
            
        except Exception as e:
            print(f"  [MiniCPM] Generation failed: {e}")
            # Fallback generation
            language = 'en'
            try:
                sample_text = ' '.join(user_posts[:2])
                if sample_text and len(sample_text) > 10:
                    language = detect(sample_text)
            except:
                pass
            
            if language == 'ja':
                return ["この話題について考えています。"]
            elif language == 'zh-cn':
                return ["思考相关的问题。"]
            else:
                return ["Thinking about current topics."]
class EnhancedMarkovChain:
    """Enhanced Markov chain method"""
    
    def __init__(self, train_df, order=2):
        self.train_df = train_df
        self.order = order
        self.chains = {}
    
    def _build_chain(self, texts):
        """Build Markov chain"""
        chain = defaultdict(list)
        
        for text in texts:
            if isinstance(text, str):
                words = text.lower().split()
                if len(words) > self.order:
                    for i in range(len(words) - self.order):
                        key = tuple(words[i:i + self.order])
                        value = words[i + self.order]
                        chain[key].append(value)
        
        return chain
    
    def generate_for_user(self, user_name, num_posts=1):
        """Generate Markov chain content"""
        user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
        
        if not user_posts:
            return ["Generated content about general topics."]
        
        if user_name not in self.chains:
            self.chains[user_name] = self._build_chain(user_posts)
        
        chain = self.chains[user_name]
        
        if not chain:
            return [f"Content for {user_name}"]
        
        start_words = random.choice(list(chain.keys()))
        words = list(start_words)
        
        for _ in range(random.randint(8, 20)):
            current_key = tuple(words[-self.order:])
            if current_key in chain and chain[current_key]:
                next_word = random.choice(chain[current_key])
                words.append(next_word)
            else:
                break
        
        sentence = ' '.join(words).capitalize()
        if not sentence.endswith(('.', '!', '?')):
            sentence += '.'
        
        return [sentence]

class SmartTemplateMethod:
    """Smart template method"""
    
    def __init__(self, train_df):
        self.train_df = train_df
        self.topic_cache = {}
    
    def generate_for_user(self, user_name, num_posts=1):
        """Generate using smart templates"""
        user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
        
        if user_name in self.topic_cache:
            topics = self.topic_cache[user_name]
        else:
            topics = []
            if user_posts:
                all_text = ' '.join(user_posts[:5])
                words = re.findall(r'\b[a-zA-Z]{4,}\b', all_text.lower())
                word_freq = Counter(words)
                common_words = {'about', 'think', 'going', 'today', 'really', 'would', 'could'}
                topics = [word for word, freq in word_freq.most_common(10) 
                         if word not in common_words and freq >= 2][:5]
            self.topic_cache[user_name] = topics
        
        templates = [
            "Following the developments on {topic} with great interest. Positive momentum building!",
            "Excellent analysis emerging about {topic}. Really appreciate the insights shared.",
            "The progress on {topic} is truly encouraging. Good work being done here.",
            "Important discussions happening around {topic}. Valuable perspectives being shared.",
            "Seeing positive movement on {topic} issues. Hopeful about the direction.",
            "Great to see engagement on {topic} topics. Constructive conversations happening.",
            "The {topic} situation shows promising developments. Worth following closely.",
            "Impressed by the depth of discussion on {topic}. Quality insights emerging.",
        ]
        
        template = random.choice(templates)
        
        if topics and '{topic}' in template:
            topic = random.choice(topics)
            post = template.replace('{topic}', topic)
        else:
            generic_topics = ['current developments', 'recent progress', 'ongoing discussions']
            topic = random.choice(generic_topics)
            post = template.replace('{topic}', topic)
        
        return [post]
class RAGBasedMethod:
    """Retrieval-augmented generation based method"""
    
    def __init__(self, train_df, model_wrapper, top_k=3):
        self.train_df = train_df
        self.model = model_wrapper
        self.top_k = top_k
    
    def generate_for_user(self, user_name, num_posts=1):
        """Generate content for user - based on retrieval-augmented generation"""
        # Get user historical posts
        user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
        
        if not user_posts:
            return ["Generated content about general topics."]
        
        # Analyze current network environment (simplified: use keywords from recent posts)
        recent_posts = self.train_df['text'].dropna().tolist()[-10:]
        env_keywords = self._extract_keywords(recent_posts)
        
        # Retrieve posts from user history relevant to current environment
        retrieved_posts = self._retrieve_posts(user_posts, env_keywords)
        
        # Use large model to generate new content based on retrieved posts
        try:
            # Build prompt
            prompt = self._build_prompt(user_name, retrieved_posts, env_keywords)
           
            user_style = self.model.analyze_user_style(user_posts[:5])
            
            # Build network environment context
            network_context = {
                'sentiment': 'neutral',
                'trending_words': env_keywords[:5],
                'recent_topics': [],
                'consensus_analysis': 'mixed opinions',
                'time_frame': 'recent'
            }
            
            # Generate post
            post = self.model.generate_context_aware_post(user_style, network_context, 0.5)
            return [post]
            
        except Exception as e:
            print(f"RAG generation failed: {e}")
            # Fallback
            return [f"Retrieved and generated content for {user_name} based on environment."]
    
    def _extract_keywords(self, posts):
        """Extract keywords"""
        all_text = ' '.join([str(p) for p in posts])
        words = re.findall(r'\b[a-zA-Z]{4,}\b', all_text.lower())
        word_freq = Counter(words)
        stopwords = {'this', 'that', 'with', 'have', 'from', 'they', 'what', 'when'}
        keywords = [word for word, freq in word_freq.most_common(20) if word not in stopwords]
        return keywords
    
    def _retrieve_posts(self, user_posts, env_keywords):
        """Retrieve posts relevant to current environment"""
        # Calculate relevance score of each post to environment keywords
        scored_posts = []
        for post in user_posts:
            if isinstance(post, str):
                score = 0
                for keyword in env_keywords[:10]:  
                    if keyword in post.lower():
                        score += 1
                scored_posts.append((score, post))
        
        # Sort by relevance, take top_k
        scored_posts.sort(key=lambda x: x[0], reverse=True)
        retrieved = [post for _, post in scored_posts[:self.top_k]]
        
        # If not enough posts retrieved, supplement with random posts
        if len(retrieved) < self.top_k:
            retrieved += user_posts[:self.top_k - len(retrieved)]
        
        return retrieved
    
    def _build_prompt(self, user_name, retrieved_posts, env_keywords):
        """Build prompt"""
        prompt = f"""Generate a social media post for user {user_name} based on their past posts and current network environment.

Past posts (for reference):
{chr(10).join(['- ' + post for post in retrieved_posts])}

Current environment keywords: {', '.join(env_keywords[:5])}

Generate a post that matches the user's style and is relevant to the current environment:
"""
        return prompt
# ========== Improved Evaluation Metrics ==========

# ========== Improved Evaluation Metrics ==========

# ========== Improved Evaluation Metrics ==========


# ========== Improved Evaluation Metrics ==========

class ImprovedEvaluationMetrics:
    """Improved evaluation metrics class"""
    
    def __init__(self, train_df, test_df):
        self.train_df = train_df
        self.test_df = test_df
        
        # Check TextBlob availability
        self._check_textblob_availability()
        
        # Extract environment features
        self.env_features = self._extract_environment_features()
        
        if self.use_textblob:
            print("  Using TextBlob for sentiment analysis")
        else:
            print("  TextBlob unavailable, using dictionary-based sentiment analysis")
    
    def _check_textblob_availability(self):
        """Check if TextBlob is available"""
        try:
            from textblob import TextBlob
            self.TextBlob = TextBlob
            self.use_textblob = True
        except ImportError:
            self.TextBlob = None
            self.use_textblob = False
    
    def _extract_environment_features(self):
        """Extract environment features - enhanced version: extract more features"""
        # Use more data
        recent_posts = self.train_df['text'].dropna().tolist()
        if len(recent_posts) > 100:
            recent_posts = recent_posts[-100:]  # Use last 100 posts
        
        if not recent_posts:
            return {'trending_words': [], 'common_topics': [], 'avg_sentiment': 0.5}
        
        all_text = ' '.join([str(p) for p in recent_posts])
        
        # Extract words
        words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower())
        word_freq = Counter(words)
        
        # Filter stopwords
        stopwords = {'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with', 'have', 'from',
                    'they', 'what', 'when', 'were', 'been', 'also', 'more', 'some', 'such',
                    'only', 'just', 'like', 'will', 'than', 'other', 'there', 'their', 'them'}
        trending_words = [word for word, freq in word_freq.most_common(30) 
                         if word not in stopwords and freq >= 2]
        
        # Extract topics (simple version: based on noun phrases)
        topics = []
        for post in recent_posts[:20]:
            if isinstance(post, str):
                # Extract uppercase word sequences (may indicate proper nouns)
                uppercase_phrases = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b', post)
                topics.extend(uppercase_phrases)
        
        topic_freq = Counter(topics)
        common_topics = [topic for topic, freq in topic_freq.most_common(10) if freq >= 2]
        
        # Calculate average sentiment
        avg_sentiment = 0.5
        if self.use_textblob:
            try:
                sentiments = []
                for post in recent_posts[:20]:
                    if isinstance(post, str) and len(post) > 10:
                        blob = self.TextBlob(post)
                        # TextBlob polarity range is [-1, 1], map to [0, 1]
                        polarity = (blob.sentiment.polarity + 1) / 2
                        sentiments.append(polarity)
                if sentiments:
                    avg_sentiment = np.mean(sentiments)
            except Exception as e:
                print(f"  Sentiment analysis failed: {e}")
        
        return {
            'trending_words': trending_words[:20],
            'common_topics': common_topics[:5],
            'avg_sentiment': avg_sentiment
        }
    
    def calculate_environment_relevance(self, generated_posts, user_name, method_name):
        """Calculate environment relevance - enhanced: multi-dimensional evaluation"""
        if not generated_posts:
            return 0.4 
        
        gen_text = ' '.join([str(p) for p in generated_posts])
        
        # 1. Vocabulary match
        gen_words = set(re.findall(r'\b[a-zA-Z]{3,}\b', gen_text.lower()))
        env_words = set(self.env_features['trending_words'])
        
        vocab_match = 0.0
        if env_words:
            common_words = gen_words & env_words
            vocab_match = len(common_words) / len(env_words) if env_words else 0
        
        # 2. Topic match
        topic_match = 0.0
        common_topics = self.env_features['common_topics']
        if common_topics:
            topic_count = 0
            for topic in common_topics:
                if topic.lower() in gen_text.lower():
                    topic_count += 1
            topic_match = topic_count / len(common_topics)
        
        # 3. Sentiment match
        sentiment_match = 0.0
        if self.use_textblob:
            try:
                blob = self.TextBlob(gen_text)
                gen_polarity = (blob.sentiment.polarity + 1) / 2  # Map to [0, 1]
                env_polarity = self.env_features['avg_sentiment']
                sentiment_match = 1 - min(abs(gen_polarity - env_polarity), 0.5) * 2
            except:
                sentiment_match = 0.5
        else:
            # Use vocabulary method to estimate sentiment
            pos_words = ['good', 'great', 'excellent', 'positive', 'optimistic', 'happy', 'progress']
            neg_words = ['bad', 'poor', 'negative', 'problem', 'issue', 'worry', 'concern']
            
            gen_words_list = gen_text.lower().split()
            pos_count = sum(1 for w in gen_words_list if w in pos_words)
            neg_count = sum(1 for w in gen_words_list if w in neg_words)
            
            gen_sentiment = 0.5
            total = pos_count + neg_count
            if total > 0:
                gen_sentiment = pos_count / total
            
            env_polarity = self.env_features['avg_sentiment']
            sentiment_match = 1 - min(abs(gen_sentiment - env_polarity), 0.5) * 2
        
        # Combined score: vocabulary (40%) + topic (30%) + sentiment (30%)
        total_score = 0.4 * vocab_match + 0.3 * topic_match + 0.3 * sentiment_match      
        
        return min(1.0, max(0.2, total_score))
    
    def calculate_style_similarity(self, generated_posts, ground_truth):
        """Calculate style similarity - comprehensive evaluation version"""
        if not generated_posts or not ground_truth:
            return 0.5
        
        # 1. Length similarity (30%)
        gen_lengths = [len(p.split()) for p in generated_posts if isinstance(p, str)]
        truth_lengths = [len(p.split()) for p in ground_truth[:5] if isinstance(p, str)]
        
        if not gen_lengths or not truth_lengths:
            length_sim = 0.5
        else:
            avg_gen = np.mean(gen_lengths)
            avg_truth = np.mean(truth_lengths)
            if avg_truth > 0:
                length_sim = 1 - min(abs(avg_gen - avg_truth) / avg_truth, 1.0)
            else:
                length_sim = 0.5
        
        # 2. Vocabulary overlap (40%)
        gen_text = ' '.join([str(p).lower() for p in generated_posts])
        truth_text = ' '.join([str(p).lower() for p in ground_truth[:5]])
        
        # Extract words (exclude short words and common words)
        gen_words = set(re.findall(r'\b[a-zA-Z]{3,}\b', gen_text))
        truth_words = set(re.findall(r'\b[a-zA-Z]{3,}\b', truth_text))
        
        common_words = {'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with', 'have', 'from',
                       'they', 'what', 'when', 'were', 'been', 'also', 'more', 'some', 'such',
                       'only', 'just', 'like', 'will', 'than', 'other', 'there', 'their', 'them'}
        
        gen_words_filtered = gen_words - common_words
        truth_words_filtered = truth_words - common_words
        
        if truth_words_filtered:
            word_overlap = len(gen_words_filtered & truth_words_filtered) / len(truth_words_filtered)
        else:
            word_overlap = 0.3  # Default value
        
        # 3. Punctuation usage similarity (15%)
        def get_punctuation_rate(text, punct):
            return text.count(punct) / len(text) if len(text) > 0 else 0
        
        punct_scores = []
        for punct in ['.', ',', '!', '?']:
            gen_rate = get_punctuation_rate(gen_text, punct)
            truth_rate = get_punctuation_rate(truth_text, punct)
            if truth_rate > 0:
                punct_sim = 1 - min(abs(gen_rate - truth_rate) / truth_rate, 1.0)
            else:
                punct_sim = 0.5 if gen_rate == 0 else 0.3
            punct_scores.append(punct_sim)
        
        punct_sim = np.mean(punct_scores) if punct_scores else 0.5
        
        # 4. Capitalization usage similarity (15%)
        def get_uppercase_ratio(text):
            letters = [c for c in text if c.isalpha()]
            return sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0
        
        gen_upper = get_uppercase_ratio(gen_text)
        truth_upper = get_uppercase_ratio(truth_text)
        
        if truth_upper > 0:
            upper_sim = 1 - min(abs(gen_upper - truth_upper) / truth_upper, 1.0)
        else:
            upper_sim = 0.5 if gen_upper == 0 else 0.3
        
        # Combined score
        final_score = (
            0.3 * length_sim + 
            0.4 * min(1.0, word_overlap * 1.5) +  # Amplify vocabulary overlap impact
            0.15 * punct_sim + 
            0.15 * upper_sim
        )
        
        return min(1.0, max(0.1, final_score))
    
    def calculate_diversity(self, generated_posts):
        """Calculate diversity - improved version"""
        if not generated_posts:
            return 0.4
        
        all_words = []
        for post in generated_posts:
            if isinstance(post, str):
                # Extract meaningful words (at least 3 letters)
                words = re.findall(r'\b[a-zA-Z]{3,}\b', post.lower())
                all_words.extend(words)
        
        if len(all_words) < 3:
            return 0.4
        
        unique_words = set(all_words)
        diversity = len(unique_words) / len(all_words)
        
        # Adjust score: if vocabulary is large enough, give bonus
        if len(all_words) >= 10:
            diversity = min(1.0, diversity * 1.2)
        
        return min(1.0, max(0.2, diversity))
    
    def calculate_habit_consistency(self, generated_posts, ground_truth):
        """Calculate habit consistency - improved version: more reasonable score range"""
        if not generated_posts or not ground_truth:
            return 0.5
        
        # Check punctuation usage habits
        def get_punctuation_stats(texts):
            if not texts:
                return {'!': 0.01, '?': 0.01, '.': 0.1, ',': 0.05, 'avg_len': 10}
            
            texts_str = [str(t) for t in texts]
            total_chars = sum(len(t) for t in texts_str)
            
            stats = {}
            for punct in ['!', '?', '.', ',']:
                count = sum(t.count(punct) for t in texts_str)
                rate = count / total_chars if total_chars > 0 else 0.01
                stats[punct] = rate
            
            # Calculate average sentence length
            sentence_lengths = []
            for text in texts_str:
                sentences = re.split(r'[.!?]+', text)
                for sentence in sentences:
                    words = sentence.strip().split()
                    if words:  # Only count non-empty sentences
                        sentence_lengths.append(len(words))
            
            avg_len = np.mean(sentence_lengths) if sentence_lengths else 10
            stats['avg_len'] = avg_len
            
            return stats
        
        gen_stats = get_punctuation_stats(generated_posts)
        truth_stats = get_punctuation_stats(ground_truth[:5])
        
        # Calculate punctuation consistency
        punct_scores = []
        for punct in ['!', '?', '.', ',']:
            gen_rate = gen_stats.get(punct, 0.01)
            truth_rate = truth_stats.get(punct, 0.01)
            
            # If both are very low (close to 0), consider them consistent
            if gen_rate < 0.02 and truth_rate < 0.02:
                punct_scores.append(1.0)
            elif truth_rate > 0:
                # Use relative difference, avoid division by very small numbers
                diff = abs(gen_rate - truth_rate)
                if diff < 0.01:  # Very small difference
                    punct_scores.append(1.0)
                else:
                    similarity = 1 - min(diff / max(truth_rate, 0.02), 1.0)
                    punct_scores.append(similarity)
            else:
                punct_scores.append(0.5)
        
        punct_consistency = np.mean(punct_scores) if punct_scores else 0.5
        
        # Length difference (more lenient penalty)
        gen_len = gen_stats.get('avg_len', 10)
        truth_len = truth_stats.get('avg_len', 10)
        
        len_diff = abs(gen_len - truth_len)
        if len_diff < 3:  # Difference less than 3 words, no penalty
            len_penalty = 0
        else:
            len_penalty = min(1.0, len_diff / 15)  # More lenient penalty
        
        # Final consistency score = punctuation consistency - length penalty
        consistency = punct_consistency - 0.2 * len_penalty  # Reduce weight of length penalty
        
        # Give base score to avoid scores being too low
        consistency = max(0.3, consistency)
        
        return min(1.0, consistency)
    
    def calculate_positivity_score(self, generated_posts):
        """Calculate positivity score - using TextBlob sentiment analysis"""
        if not generated_posts:
            return 0.5
        
        all_text = ' '.join([str(p) for p in generated_posts])
        
        # Use TextBlob for sentiment analysis
        if self.use_textblob:
            try:
                blob = self.TextBlob(all_text)
                # TextBlob polarity range is [-1, 1], map to [0, 1]
                polarity = blob.sentiment.polarity
                positivity = (polarity + 1) / 2
                
                # Consider subjectivity (confidence)
                subjectivity = blob.sentiment.subjectivity
                
                # Adjust score: if there is clear emotional tendency (high subjectivity), give slight bonus
                if subjectivity > 0.5 and positivity > 0.5:
                    positivity = min(1.0, positivity * 1.1)
                elif subjectivity > 0.5 and positivity < 0.5:
                    positivity = max(0.0, positivity * 0.9)
                
                return min(1.0, max(0.0, positivity))
                
            except Exception as e:
                print(f"  TextBlob sentiment analysis failed: {e}, using dictionary-based method")
                # Fallback to vocabulary method
        
        # Vocabulary method (fallback)
        words = all_text.lower().split()
        
        # Extended positive word list
        positive_words = [
            'good', 'great', 'excellent', 'positive', 'optimistic', 
            'happy', 'progress', 'success', 'improve', 'better',
            'love', 'like', 'support', 'agree', 'encourage', 'wonderful',
            'amazing', 'fantastic', 'brilliant', 'outstanding', 'superb',
            'awesome', 'terrific', 'perfect', 'fabulous', 'marvelous',
            'exceptional', 'commend', 'praise', 'appreciate', 'thank',
            'grateful', 'pleased', 'delighted', 'satisfied', 'content',
            'joy', 'excited', 'enthusiastic', 'hopeful', 'promising'
        ]
        
        # Extended negative word list
        negative_words = [
            'bad', 'poor', 'negative', 'problem', 'issue', 'worry',
            'concern', 'difficult', 'challenge', 'failure', 'hate',
            'dislike', 'against', 'disagree', 'criticize', 'terrible',
            'awful', 'horrible', 'disappoint', 'frustrate', 'angry',
            'upset', 'sad', 'unhappy', 'dissatisfied', 'complain',
            'critique', 'blame', 'fault', 'wrong', 'error', 'mistake',
            'fail', 'weak', 'poorly', 'inadequate', 'insufficient',
            'problematic', 'trouble', 'difficulty', 'struggle'
        ]
        
        pos_count = sum(1 for word in words if word in positive_words)
        neg_count = sum(1 for word in words if word in negative_words)
        
        total = pos_count + neg_count
        if total > 0:
            positivity = pos_count / total
        else:
            # If no sentiment words, check for sentiment phrases
            positive_phrases = ['well done', 'good job', 'nice work', 'keep it up', 'looking good']
            negative_phrases = ['not good', 'bad idea', 'poor quality', 'needs improvement']
            
            pos_phrase_count = sum(1 for phrase in positive_phrases if phrase in all_text.lower())
            neg_phrase_count = sum(1 for phrase in negative_phrases if phrase in all_text.lower())
            
            if pos_phrase_count > neg_phrase_count:
                positivity = 0.7
            elif neg_phrase_count > pos_phrase_count:
                positivity = 0.3
            else:
                positivity = 0.5
        
        # Adjust: if there are positive words and no negative words, increase score
        if pos_count > 0 and neg_count == 0:
            positivity = min(1.0, positivity * 1.2)
        # If there are negative words and no positive words, decrease score
        elif neg_count > 0 and pos_count == 0:
            positivity = max(0.0, positivity * 0.8)
        
        return min(1.0, max(0.0, positivity))
    
    def calculate_realness_score(self, generated_posts, ground_truth):
        """Calculate realness score - improved version"""
        if not generated_posts or not ground_truth:
            return 0.5
        
        # Check multiple dimensions of authenticity
        scores = []
        
        # 1. Length consistency (30%)
        gen_lengths = [len(p.split()) for p in generated_posts if isinstance(p, str)]
        truth_lengths = [len(p.split()) for p in ground_truth[:5] if isinstance(p, str)]
        
        if gen_lengths and truth_lengths:
            avg_gen = np.mean(gen_lengths)
            avg_truth = np.mean(truth_lengths)
            if avg_truth > 0:
                length_sim = 1 - min(abs(avg_gen - avg_truth) / avg_truth, 1.0)
            else:
                length_sim = 0.5
            scores.append(length_sim * 0.3)
        else:
            scores.append(0.15)
        
        # 2. Vocabulary complexity match (30%)
        def get_vocab_complexity(texts):
            if not texts:
                return 0.5
            all_text = ' '.join([str(t) for t in texts])
            words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower())
            if not words:
                return 0.5
            # Long word ratio as complexity indicator
            long_words = [w for w in words if len(w) >= 6]
            return len(long_words) / len(words)
        
        gen_complexity = get_vocab_complexity(generated_posts)
        truth_complexity = get_vocab_complexity(ground_truth[:5])
        
        complexity_sim = 1 - min(abs(gen_complexity - truth_complexity), 0.5) * 2
        scores.append(complexity_sim * 0.3)
        
        # 3. Structure similarity (40%) - check sentence beginning patterns
        def get_start_patterns(texts, n=3):
            patterns = []
            for text in texts[:3]:
                if isinstance(text, str) and text.strip():
                    first_word = text.strip().split()[0].lower() if text.strip().split() else ""
                    patterns.append(first_word)
            return patterns
        
        gen_starts = get_start_patterns(generated_posts)
        truth_starts = get_start_patterns(ground_truth[:3])
        
        if truth_starts and gen_starts:
            # Check if there are common starting words
            common_starts = set(gen_starts) & set(truth_starts)
            if common_starts:
                start_sim = len(common_starts) / max(len(set(truth_starts)), 1)
            else:
                # Check starting word types (e.g., pronouns, verbs, etc.)
                pronoun_starts = {'i', 'you', 'he', 'she', 'we', 'they', 'it'}
                gen_pronoun = any(start in pronoun_starts for start in gen_starts)
                truth_pronoun = any(start in pronoun_starts for start in truth_starts)
                start_sim = 0.7 if gen_pronoun == truth_pronoun else 0.3
        else:
            start_sim = 0.5
        
        scores.append(start_sim * 0.4)
        
        final_score = sum(scores)
        return min(1.0, max(0.2, final_score))
# ========== Experiment Pipeline ==========


# In the experiment pipeline, modify the methods list
# In the run_single_experiment function, modify the methods list:

def run_single_experiment(train_df, test_df, user_name, model_wrapper, run_id=0, recent_hours=48):
    """Run a single experiment"""
    # Check if user has sufficient data
    user_posts = train_df[train_df['userName'] == user_name]['text'].dropna().tolist()
    if len(user_posts) < 3:
        print(f"  User {user_name} has insufficient data, skipping")
        return {}, {}
    
    print(f"\n  {'='*60}")
    print(f"  User: {user_name}")
    print(f"  Data volume: {len(user_posts)} posts")
    print(f"  {'='*60}")
    
    # Initialize methods
    methods = {
        'intelligent_SCL_method': IntelligentSCLMethod(train_df, model_wrapper, 
                                                      positivity_boost=0.5, 
                                                      n_particles=5,
                                                      recent_hours=recent_hours),
        'miniCPM_only': MiniCPMOnlyMethod(train_df, model_wrapper),
        'markov_chain': EnhancedMarkovChain(train_df),
        'smart_template': SmartTemplateMethod(train_df),
        'cgan_method': CGANMethod(train_df, epochs=5),  # Reduce training epochs
        'twice': TWICEMethod(train_df, model_wrapper),   # New TWICE baseline
    }
    
    # Evaluation metrics
    evaluator = ImprovedEvaluationMetrics(train_df, test_df)
    
    # Get ground truth
    ground_truth = test_df[
        (test_df['userName'] == user_name) & 
        (test_df['text'].notna())
    ]['text'].tolist()[:5]
    
    # Ensure all variables are initialized before try block
    results = {}
    generated_outputs = {}
    
    print(f"\n  [Experiment Run {run_id+1}] Starting generation...")
    print(f"  {'─'*40}")
    
    for method_name, method in methods.items():
        try:
            print(f"    ▶ {method_name}: ", end="")
            
            # Record start time
            start_time = time.time()
            
            # Generate posts
            generated_posts = method.generate_for_user(user_name, 1)
            
            # Record end time
            elapsed_time = time.time() - start_time
            
            # Ensure there is output
            if not generated_posts or len(generated_posts) == 0:
                print(f"✗ Generation result is empty")
                generated_posts = [f"Generated by {method_name}"]
            
            # Ensure each element is a string
            generated_posts = [str(post) for post in generated_posts]
            
            # Store generated sentences
            generated_outputs[method_name] = generated_posts[0] if generated_posts else "Generation failed"
            
            # Display generation result and time
            if isinstance(generated_outputs[method_name], str) and generated_outputs[method_name].startswith("Generation failed"):
                print(f"✗ ({elapsed_time:.2f}s) Generation failed")
            else:
                # Truncate overly long text
                display_text = generated_outputs[method_name]
                if len(display_text) > 80:
                    display_text = display_text[:77] + "..."
                print(f"✓ ({elapsed_time:.2f}s) {display_text}")
            
            # Calculate metrics
            metrics = {
                'diversity': evaluator.calculate_diversity(generated_posts),
                'style_similarity': evaluator.calculate_style_similarity(generated_posts, ground_truth),
                'environment_relevance': evaluator.calculate_environment_relevance(generated_posts, user_name, method_name),
                'habit_consistency': evaluator.calculate_habit_consistency(generated_posts, ground_truth),
                'positivity_score': evaluator.calculate_positivity_score(generated_posts),
                'realness_score': evaluator.calculate_realness_score(generated_posts, ground_truth)
            }
            
            results[method_name] = {
                'posts': generated_posts,
                'metrics': metrics,
                'run_id': run_id,
                'time_elapsed': elapsed_time
            }
            
        except Exception as e:
            error_msg = str(e)[:50]
            print(f"✗ Failed: {error_msg}")
            
            # Ensure generated_outputs is already defined
            if 'generated_outputs' not in locals():
                generated_outputs = {}
                
            generated_outputs[method_name] = f"Generation failed: {error_msg}"
            
            # Ensure results is already defined
            if 'results' not in locals():
                results = {}
                
            results[method_name] = {
                'posts': ["Generation failed"],
                'metrics': {k: 0.5 for k in ['diversity', 'style_similarity', 'environment_relevance', 
                                           'habit_consistency', 'positivity_score', 'realness_score']},
                'run_id': run_id,
                'time_elapsed': 0
            }
    
    # Display summary of generated results for all methods
    print(f"\n  {'─'*40}")
    print("  Generation Results Summary:")
    print(f"  {'─'*40}")
    for method_name, output in generated_outputs.items():
        status = "✓" if not output.startswith("Generation failed") else "✗"
        print(f"    {status} {method_name:25s}: {output[:60]}{'...' if len(output) > 60 else ''}")
    
    return results, generated_outputs
    
def process_all_datasets(n_repeats=3, recent_hours=48):
    """Process all datasets"""
    print("Intelligent Social Network-Aware User Cloning Experimental System")
    print("="*60)
    
    print("Note: torch library is required for CGAN training")
    print("Install command: pip install torch")
    print("="*60)
    
    print("Note: langdetect library is required for language detection")
    print("Install command: pip install langdetect")
    print("="*60)
    
    try:
        model_wrapper = SmartModelWrapper()
    except Exception as e:
        print(f"Model loading failed: {e}")
        print("Will run in simplified mode (some functions limited)")
        model_wrapper = None
        return
    
    excel_files = glob.glob("*.xlsx")
    print(f"Found {len(excel_files)} datasets")
    
    if not excel_files:
        print("No Excel files found")
        return
    
    all_results = {}
    
    for file_path in excel_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        print(f"\nProcessing dataset: {dataset_name}")
        
        try:
            df = pd.read_excel(file_path, engine="openpyxl")
            print(f"Data rows: {len(df)}")
            
           
            
            # main_user = user_counts.index[0]
            # frequency# degree centrality
            USER_SELECT_MODE = "frequency"
            # ==========================================

            user_counts = df['userName'].value_counts()
            if len(user_counts) == 0:
                print("No valid users")
                continue

            if USER_SELECT_MODE == "frequency":            
                main_user = user_counts.index[0]
            elif USER_SELECT_MODE == "degree centrality":
                # Select user with highest degree centrality
                # follow_edges is the follower edge table for the current dataset, columns: source(follower), target(followed)
                G = nx.DiGraph()
                G.add_edges_from(zip(follow_edges["source"], follow_edges["target"]))
                bc = nx.degree_centrality(G, normalized=True)
                main_user = max(bc, key=bc.get)
            user_post_count = user_counts[main_user]
            print(f"Main user: {main_user} (post count: {user_post_count})")
            
            if user_post_count < 4:
                print(f"User {main_user} has insufficient data ({user_post_count} posts), skipping")
                continue
            
            user_data = df[df['userName'] == main_user].reset_index(drop=True)
            
            split_idx = len(user_data) // 2
            train_df = user_data.iloc[:split_idx].copy()
            test_df = user_data.iloc[split_idx:].copy()
            
            print(f"Training set: {len(train_df)} posts, Test set: {len(test_df)} posts")
            
            dataset_results = []
            
            # Modify here: remove min restriction, run n_repeats times directly
            for run_id in range(n_repeats):  # Before: range(min(n_repeats, 1))
                print(f"\n  Experiment Run {run_id+1}/{n_repeats}:")
                
                # Set random seed for reproducible but different results
                random.seed(run_id * 1000 + 42)  # Set different seed for each run
                np.random.seed(run_id * 1000 + 42)
                
                results, generated_outputs = run_single_experiment(train_df, test_df, main_user, 
                                                                  model_wrapper, run_id, recent_hours)
                
                if not results:
                    print("  No valid results, skipping")
                    continue
                
                print("  Generated post content:")
                for method_name, output in generated_outputs.items():
                    print(f"    {method_name}: {output}")
                
                dataset_results.append(results)
            
            if dataset_results:
                all_results[dataset_name] = {
                    'user': main_user,
                    'train_size': len(train_df),
                    'test_size': len(test_df),
                    'results': dataset_results
                }
                
                save_immediate_results(dataset_name, main_user, dataset_results)
            
        except Exception as e:
            print(f"Error processing dataset {dataset_name}: {e}")
            import traceback
            traceback.print_exc()
    
    if all_results:
        summarize_all_results(all_results, n_repeats)

def save_immediate_results(dataset_name, user_name, dataset_results):
    """Save results to CSV file"""
    output_dir = "experiment_outputs"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    rows = []
    
    for run_id, run_results in enumerate(dataset_results):
        for method_name, method_data in run_results.items():
            row = {
                'dataset': dataset_name,
                'user': user_name,
                'run_id': run_id,
                'method': method_name,
                'generated_post': method_data['posts'][0] if method_data['posts'] else ''
            }
            
            for metric_name, value in method_data['metrics'].items():
                row[metric_name] = value
            
            rows.append(row)
    
    df = pd.DataFrame(rows)
    output_file = os.path.join(output_dir, f"{dataset_name}_intelligent_results.csv")
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Results saved to: {output_file}")

def summarize_all_results(all_results, n_repeats):
    # ...
    summary_rows = []
    
    # Collect all method names that appear
    all_method_names = set()
    for dataset_info in all_results.values():
        for run_results in dataset_info['results']:
            all_method_names.update(run_results.keys())
    # If fixed order is desired, can sort or manually specify
    all_method_names = sorted(list(all_method_names))  # Sort alphabetically
    
    for dataset_name, dataset_info in all_results.items():
        user_name = dataset_info['user']
        dataset_results = dataset_info['results']
        
        for method_name in all_method_names:   # Use dynamic list
            method_metrics = {metric: [] for metric in ['diversity', 'style_similarity', 'environment_relevance', 
                                                       'habit_consistency', 'positivity_score', 'realness_score']}
            # ... Collect metrics ...
            
            for run_results in dataset_results:
                if method_name in run_results:
                    for metric_name, value in run_results[method_name]['metrics'].items():
                        method_metrics[metric_name].append(value)
            
            stats = {}
            for metric_name, values in method_metrics.items():
                if values:
                    stats[f'{metric_name}_mean'] = np.mean(values)
                    stats[f'{metric_name}_std'] = np.std(values)
                else:
                    stats[f'{metric_name}_mean'] = 0.5
                    stats[f'{metric_name}_std'] = 0.0
            
            row = {
                'dataset': dataset_name,
                'user': user_name,
                'method': method_name,
                'n_repeats': n_repeats
            }
            row.update(stats)
            summary_rows.append(row)
    
    output_dir = "final_results"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    summary_df = pd.DataFrame(summary_rows)
    summary_file = os.path.join(output_dir, "intelligent_summary.csv")
    summary_df.to_csv(summary_file, index=False)
    print(f"Summary results saved to: {summary_file}")
    
    # Performance analysis
    # print("\nMethod Performance Comparison Analysis:")
    # print("-"*100)
    
    # # methods = ['intelligent_SCL_method', 'miniCPM_only', 'markov_chain', 'smart_template']
    # methods = sorted([m for m in all_method_names if m in summary_df['method'].unique()])
    
    # for method in methods:
    #     method_data = summary_df[summary_df['method'] == method]
    #     if len(method_data) > 0:
    #         print(f"\n{method}:")
    #         print(f"  Environment Relevance: {method_data['environment_relevance_mean'].mean():.3f} ± {method_data['environment_relevance_std'].mean():.3f}")
    #         print(f"  Positivity Score: {method_data['positivity_score_mean'].mean():.3f} ± {method_data['positivity_score_std'].mean():.3f}")
    #         print(f"  Style Similarity: {method_data['style_similarity_mean'].mean():.3f} ± {method_data['style_similarity_std'].mean():.3f}")
    #         print(f"  Habit Consistency: {method_data['habit_consistency_mean'].mean():.3f} ± {method_data['habit_consistency_std'].mean():.3f}")
    
    # Improvement effect evaluation
    # print("\nImprovement Effect Evaluation:")
    # print("-"*100)
    intelligent_data = summary_df[summary_df['method'] == 'intelligent_SCL_method']
    minicpm_data = summary_df[summary_df['method'] == 'miniCPM_only']
    
    if len(intelligent_data) > 0 and len(minicpm_data) > 0:
        env_improvement = intelligent_data['environment_relevance_mean'].mean() - minicpm_data['environment_relevance_mean'].mean()
        pos_improvement = intelligent_data['positivity_score_mean'].mean() - minicpm_data['positivity_score_mean'].mean()
        
        print(f"Environment Relevance Improvement: {env_improvement:.3f}")
        print(f"Positivity Score Improvement: {pos_improvement:.3f}")
        
        if env_improvement > 0.1:
            print("✓ Significant improvement: intelligent_SCL_method significantly outperforms miniCPM_only in environment relevance")
        elif env_improvement > 0.05:
            print("✓ Partial success: intelligent_SCL_method outperforms miniCPM_only in environment relevance")
        elif env_improvement > 0:
            print("○ Slight improvement: intelligent_SCL_method slightly outperforms miniCPM_only in environment relevance")
        elif env_improvement < -0.1:
            print("✗ Significant regression: intelligent_SCL_method significantly underperforms miniCPM_only in environment relevance")
        else:
            print("○ Essentially equal: intelligent_SCL_method is comparable to miniCPM_only in environment relevance")

# ========== Main Function ==========


if __name__ == "__main__":
    # Set parameters
    N_REPEATS = 5  # Before: N_REPEATS = 1
    RECENT_HOURS = 48  # Define the "recent" time window (hours)
    
    print(f"Experiment Configuration:")
    print(f"- Number of repeats: {N_REPEATS}")
    print(f"- Time window: last {RECENT_HOURS} hours")
    print(f"- Intelligent user style learning and environment integration")
    print("="*60)
    
    # Check dependencies
    try:
        import langdetect
        print("✓ langdetect library is installed")
    except ImportError:
        print("✗ langdetect library is not installed, please run: pip install langdetect")
        exit(1)
    
    process_all_datasets(n_repeats=N_REPEATS, recent_hours=RECENT_HOURS)

TextBlob successfully imported
Experiment Configuration:
- Number of repeats: 5
- Time window: last 48 hours
- Intelligent user style learning and environment integration
✓ langdetect library is installed
Intelligent Social Network-Aware User Cloning Experimental System
Note: torch library is required for CGAN training
Install command: pip install torch
Note: langdetect library is required for language detection
Install command: pip install langdetect
Loading model: C:/Users/fs/Desktop/lwz/models/MiniCPM-4.0-8B
Using GPU: NVIDIA GeForce RTX 4090 Memory: 25.8 GB


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully
Model device: cuda:0
Found 29 datasets

Processing dataset: #AI
Data rows: 1284
Main user: darrello (post count: 45)
Training set: 22 posts, Test set: 23 posts

  Experiment Run 1/5:

  User: darrello
  Data volume: 22 posts
  Successfully parsed timestamp column, earliest: 2020-03-14 19:31:00, latest: 2025-11-21 01:13:59
  Building recent social network, active users: 1
  Network construction complete, nodes: 1, edges: 0
  Using 5 time-aware particles to explore recent network environment...
  Particle exploration complete, collected 15 recent posts
  Intelligent SCL method initialization complete:
    - Time window: last 48 hours
    - Recent post count: 3
    - Active user count: 1
    - Trending words: ['https', 'live', 'session', 'lfdecentralized', 'governance']
  CGAN training data: 22 texts, 1 users
  CGAN method initialized, device: cuda, training epochs: 5
  Using TextBlob for sentiment analysis

  [Experiment Run 1] Starting generation...
  ─────────

Traceback (most recent call last):
  File "C:\anaconda\envs\minicpm_env\lib\site-packages\pandas\core\indexes\base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'userName'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\fs\AppData\Local\Temp\ipykernel_62796\2922438904.py", line 2125, in process_all_datasets
    user_counts = df['userName'].value_counts()
  File "C:\anaconda\envs\minicpm_env\lib\site-packages\pandas\core\frame.py", line 4113, in __getitem__
    indexer = self.columns.get_

In [1]:
class RetrievalAugmentedGenerator:
    """Retrieval-augmented generation method - retrieve relevant content based on user history and network environment and generate"""
    
    def __init__(self, train_df, model_wrapper=None, top_k=3):
        """
        Initialize retrieval-augmented generator
        
        Args:
            train_df: Training data
            model_wrapper: Large model wrapper (optional)
            top_k: Number of most similar samples to retrieve
        """
        self.train_df = train_df
        self.model_wrapper = model_wrapper
        self.top_k = top_k
        
        # Build TF-IDF vectorizer for retrieval
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2
        )
        
        # Prepare retrieval index
        self._build_retrieval_index()
        
        # For environment topic modeling
        self.env_topics = {}
    
    def _build_retrieval_index(self):
        """Build retrieval index"""
        # Extract all texts
        all_texts = self.train_df['text'].dropna().tolist()
        if len(all_texts) < 10:
            print("  Warning: Limited data volume, retrieval effectiveness may be restricted")
        
        # Build TF-IDF matrix
        self.tfidf_matrix = self.vectorizer.fit_transform(all_texts)
        self.text_ids = list(range(len(all_texts)))
        
        # Cache text content
        self.text_cache = all_texts
        
        print(f"  Retrieval index construction complete, total {len(all_texts)} texts")
    
    def _retrieve_similar_texts(self, query_text, user_name=None, top_k=None):
        """Retrieve similar texts"""
        if top_k is None:
            top_k = self.top_k
        
        if len(self.text_cache) == 0:
            return []
        
        # Vectorize query text
        query_vec = self.vectorizer.transform([query_text])
        
        # Calculate cosine similarity
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        
        # Get indices of most similar texts
        similar_indices = similarities.argsort()[-top_k:][::-1]
        
        # Filter by similarity threshold
        similar_texts = []
        for idx in similar_indices:
            if similarities[idx] > 0.1:  # Similarity threshold
                text = self.text_cache[idx]
                # If username is specified, prioritize retrieving that user's texts
                if user_name:
                    try:
                        # Get original data index
                        original_idx = self.train_df[self.train_df['text'] == text].index[0]
                        if self.train_df.loc[original_idx, 'userName'] == user_name:
                            # User's own text, give higher priority
                            similar_texts.append((text, similarities[idx] * 1.5))
                        else:
                            similar_texts.append((text, similarities[idx]))
                    except:
                        similar_texts.append((text, similarities[idx]))
                else:
                    similar_texts.append((text, similarities[idx]))
        
        # Sort by similarity
        similar_texts.sort(key=lambda x: x[1], reverse=True)
        
        return [text for text, _ in similar_texts[:top_k]]
    
    def _extract_environment_context(self, recent_hours=24):
        """Extract environment context"""
        # Get recent data
        if 'parsed_time' in self.train_df.columns:
            latest_time = self.train_df['parsed_time'].max()
            time_threshold = latest_time - pd.Timedelta(hours=recent_hours)
            recent_data = self.train_df[self.train_df['parsed_time'] >= time_threshold]
        else:
            recent_data = self.train_df.tail(50)
        
        # Extract trending words
        recent_texts = recent_data['text'].dropna().tolist()
        if not recent_texts:
            return {"trending_topics": [], "sentiment": "neutral"}
        
        all_recent_text = ' '.join(recent_texts)
        
        # Use TF-IDF to extract keywords
        tfidf = TfidfVectorizer(max_features=20, stop_words='english')
        tfidf_matrix = tfidf.fit_transform([all_recent_text])
        feature_names = tfidf.get_feature_names_out()
        
        # Get highest weighted words
        weights = tfidf_matrix.toarray()[0]
        top_indices = weights.argsort()[-10:][::-1]
        trending_topics = [feature_names[i] for i in top_indices if weights[i] > 0]
        
        # Analyze sentiment
        if self.model_wrapper and hasattr(self.model_wrapper, 'analyze_user_style'):
            try:
                sentiment = self.model_wrapper._analyze_emotional_tone(recent_texts[:10])
            except:
                sentiment = "neutral"
        else:
            # Simple sentiment analysis
            positive_words = ['good', 'great', 'excellent', 'positive', 'happy']
            negative_words = ['bad', 'poor', 'negative', 'problem', 'issue']
            
            pos_count = sum(1 for text in recent_texts[:10] 
                          if any(word in text.lower() for word in positive_words))
            neg_count = sum(1 for text in recent_texts[:10] 
                          if any(word in text.lower() for word in negative_words))
            
            if pos_count > neg_count:
                sentiment = "positive"
            elif neg_count > pos_count:
                sentiment = "negative"
            else:
                sentiment = "neutral"
        
        return {
            "trending_topics": trending_topics,
            "sentiment": sentiment,
            "recent_posts_sample": recent_texts[:3]
        }
    
    def generate_for_user(self, user_name, num_posts=1, use_llm=True):
        """Generate content for user - retrieval-augmented method"""
        # Get user history
        user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
        
        if not user_posts:
            return [f"Content generated for {user_name}"]
        
        # Extract environment context
        env_context = self._extract_environment_context()
        
        if use_llm and self.model_wrapper:
            # Method 1: Use large model for generation
            return self._generate_with_llm(user_name, user_posts, env_context, num_posts)
        else:
            # Method 2: Retrieval-based generation (no large model needed)
            return self._generate_with_retrieval(user_name, user_posts, env_context, num_posts)
    
    def _generate_with_llm(self, user_name, user_posts, env_context, num_posts):
        """Generate using large model"""
        try:
            # Analyze user style
            user_style = self.model_wrapper.analyze_user_style(user_posts)
            
            # Build network environment context
            network_context = {
                'sentiment': env_context['sentiment'],
                'trending_words': env_context['trending_topics'],
                'recent_topics': env_context['trending_topics'][:3],
                'consensus_analysis': f"{env_context['sentiment']} consensus",
                'time_frame': 'recent 24 hours',
                'data_source': 'retrieval_augmented'
            }
            
            # Retrieve similar content as reference
            query_text = ' '.join(user_posts[:2]) if user_posts else "social media post"
            retrieved_texts = self._retrieve_similar_texts(query_text, user_name, top_k=2)
            
            # If relevant content is retrieved, add to prompt
            if retrieved_texts:
                retrieval_context = "\nRetrieved similar posts for reference:\n"
                for i, text in enumerate(retrieved_texts[:2], 1):
                    retrieval_context += f"{i}. {text}\n"
                
                # Modify user style, add retrieved content
                if 'post_samples' in user_style:
                    user_style['post_samples'] = user_style['post_samples'][:1] + retrieved_texts[:1]
                else:
                    user_style['post_samples'] = retrieved_texts[:2]
            
            # Generate posts
            posts = []
            for _ in range(num_posts):
                post = self.model_wrapper.generate_context_aware_post(
                    user_style, 
                    network_context, 
                    positivity_boost=0.6
                )
                posts.append(post)
            
            return posts
            
        except Exception as e:
            print(f"  LLM generation failed: {e}")
            # Fallback to retrieval generation
            return self._generate_with_retrieval(user_name, user_posts, env_context, num_posts)
    
    def _generate_with_retrieval(self, user_name, user_posts, env_context, num_posts):
        """Retrieval-based generation (no large model needed)"""
        # Extract user writing features
        user_features = self._extract_user_features(user_posts)
        
        # Build query: combine user features and environment features
        if env_context['trending_topics']:
            query_text = f"{' '.join(user_features['common_words'][:3])} {env_context['trending_topics'][0]}"
        else:
            query_text = ' '.join(user_posts[:1]) if user_posts else "general discussion"
        
        # Retrieve similar content
        retrieved_texts = self._retrieve_similar_texts(query_text, user_name, top_k=5)
        
        if not retrieved_texts:
            # If no content retrieved, use template-based fallback
            return self._generate_with_template(user_features, env_context, num_posts)
        
        # Adapt based on retrieval results
        generated_posts = []
        for _ in range(num_posts):
            base_text = random.choice(retrieved_texts)
            modified_text = self._adapt_text(base_text, user_features, env_context)
            generated_posts.append(modified_text)
        
        return generated_posts
    
    def _extract_user_features(self, user_posts):
        """Extract user features"""
        if not user_posts:
            return {
                'common_words': [],
                'avg_length': 15,
                'language': 'en'
            }
        
        # Extract common words
        all_text = ' '.join([p.lower() for p in user_posts if isinstance(p, str)])
        words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text)
        
        stopwords = {'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with', 'have', 'from'}
        filtered_words = [w for w in words if w not in stopwords]
        
        word_freq = Counter(filtered_words)
        common_words = [word for word, _ in word_freq.most_common(10)]
        
        # Calculate average length
        lengths = [len(p.split()) for p in user_posts if isinstance(p, str)]
        avg_length = np.mean(lengths) if lengths else 15
        
        # Detect language
        language = 'en'
        try:
            sample_text = ' '.join(user_posts[:2])
            language = detect(sample_text) if len(sample_text) > 20 else 'en'
        except:
            pass
        
        return {
            'common_words': common_words,
            'avg_length': avg_length,
            'language': language,
            'sample_posts': user_posts[:3]
        }
    
    def _adapt_text(self, base_text, user_features, env_context):
        """Adapt text to match user style and environment"""
        if not isinstance(base_text, str):
            return base_text
        
        # Simple text adaptation strategy
        words = base_text.split()
        
        # 1. Adjust length to match user habits
        target_length = int(user_features['avg_length'])
        if len(words) > target_length * 1.5:
            words = words[:target_length]
        elif len(words) < target_length * 0.7 and len(words) > 5:
            # Can add some user common words
            if user_features['common_words']:
                add_word = random.choice(user_features['common_words'])
                words.append(add_word)
        
        # 2. If environment has trending topics, try to incorporate
        if env_context['trending_topics'] and random.random() > 0.5:
            topic = random.choice(env_context['trending_topics'][:3])
            # Ensure topic is not in text
            if topic.lower() not in ' '.join(words).lower():
                if random.random() > 0.7:
                    words.insert(random.randint(0, len(words)//2), topic)
                else:
                    words.append(f"#{topic}")
        
        # 3. Adjust sentiment tendency
        if env_context['sentiment'] == 'positive':
            # Add positive words
            positive_words = ['great', 'good', 'interesting', 'exciting', 'progress']
            if random.random() > 0.6:
                words.append(random.choice(positive_words))
        
        # Recombine text
        adapted_text = ' '.join(words)
        
        # Ensure sentence ends with punctuation
        if not adapted_text.endswith(('.', '!', '?')):
            adapted_text += '.'
        
        return adapted_text.capitalize()
    
    def _generate_with_template(self, user_features, env_context, num_posts):
        """Template-based generation (fallback)"""
        templates_en = [
            "Following the discussion on {topic}. {user_word} insights emerging.",
            "Interesting perspectives shared about {topic}. Worth considering.",
            "The {topic} conversation continues to evolve. Good points being made.",
            "Reflecting on the {topic} developments. Positive momentum building.",
            "Engaging with the {topic} discourse. Valuable exchange of ideas."
        ]
        
        templates_zh = [
            "关于{topic}的讨论很有启发。{user_word}观点值得关注。",
            "{topic}话题持续引发思考。有意义的交流正在进行。",
            "关注{topic}的最新进展。积极的讨论氛围。",
            "对{topic}的探讨不断深入。建设性的意见交流。"
        ]
        
        # Select template
        if user_features['language'] == 'zh-cn':
            templates = templates_zh
        else:
            templates = templates_en
        
        # Prepare fill content
        if env_context['trending_topics']:
            topic = random.choice(env_context['trending_topics'])
        else:
            topic = "current developments"
        
        user_word = ""
        if user_features['common_words']:
            user_word = random.choice(user_features['common_words'][:3])
        
        generated_posts = []
        for _ in range(num_posts):
            template = random.choice(templates)
            post = template.format(topic=topic, user_word=user_word)
            generated_posts.append(post)
        
        return generated_posts


class NeuralTemplateMethod:
    """Neural template method - use neural network to learn generation patterns"""
    
    def __init__(self, train_df, model_wrapper=None):
        self.train_df = train_df
        self.model_wrapper = model_wrapper
        self.user_patterns = {}
        self._learn_generation_patterns()
    
    def _learn_generation_patterns(self):
        """Learn user generation patterns"""
        print("  Learning user generation patterns...")
        
        # Group by user
        user_groups = self.train_df.groupby('userName')
        
        for user_name, group in user_groups:
            user_posts = group['text'].dropna().tolist()
            if len(user_posts) < 3:
                continue
            
            # Analyze user patterns
            patterns = self._analyze_user_patterns(user_posts)
            self.user_patterns[user_name] = patterns
        
        print(f"  Learned generation patterns for {len(self.user_patterns)} users")
    
    def _analyze_user_patterns(self, user_posts):
        """Analyze user posting patterns"""
        patterns = {
            'sentence_structure': [],
            'common_phrases': [],
            'content_patterns': [],
            'topic_distribution': []
        }
        
        # Analyze sentence structure
        for post in user_posts[:10]:
            if isinstance(post, str):
                # Check sentence beginning
                first_word = post.split()[0].lower() if post.split() else ""
                if first_word in ['i', 'we', 'you', 'the', 'this', 'that']:
                    patterns['sentence_structure'].append(f"starts_with_{first_word}")
                
                # Check sentence length
                word_count = len(post.split())
                if word_count < 10:
                    patterns['sentence_structure'].append("short_sentence")
                elif word_count > 25:
                    patterns['sentence_structure'].append("long_sentence")
        
        # Extract common phrases
        all_text = ' '.join([p.lower() for p in user_posts[:5]])
        # Extract 2-3 word phrases
        words = all_text.split()
        for i in range(len(words) - 2):
            phrase = ' '.join(words[i:i+3])
            if len(phrase) > 10 and phrase.count(' ') == 2:
                patterns['common_phrases'].append(phrase)
        
        # Simplify: take top 3 most common patterns
        patterns['sentence_structure'] = list(set(patterns['sentence_structure']))[:3]
        patterns['common_phrases'] = list(set(patterns['common_phrases']))[:5]
        
        return patterns
    
    def generate_for_user(self, user_name, num_posts=1):
        """Generate content based on learned patterns"""
        if user_name not in self.user_patterns:
            # If no pattern learned, use simple generation
            user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
            if not user_posts:
                return ["Generated content based on general patterns."]
            
            # Temporarily analyze patterns
            patterns = self._analyze_user_patterns(user_posts)
        else:
            patterns = self.user_patterns[user_name]
        
        # Generate based on patterns
        generated_posts = []
        for _ in range(num_posts):
            post = self._generate_from_patterns(patterns)
            generated_posts.append(post)
        
        return generated_posts
    
    def _generate_from_patterns(self, patterns):
        """Generate content based on patterns"""
        # Simple pattern-driven generation
        sentence_structures = patterns.get('sentence_structure', [])
        common_phrases = patterns.get('common_phrases', [])
        
        # Select sentence structure
        if sentence_structures:
            structure = random.choice(sentence_structures)
        else:
            structure = "starts_with_i"
        
        # Basic templates
        if structure == "starts_with_i":
            templates = [
                "I think {phrase} is worth discussing.",
                "I've been following {phrase} with interest.",
                "I appreciate the insights on {phrase}."
            ]
        elif structure == "starts_with_the":
            templates = [
                "The discussion about {phrase} continues.",
                "The progress on {phrase} is noticeable.",
                "The insights shared about {phrase} are valuable."
            ]
        else:
            templates = [
                "Interesting points about {phrase}.",
                "Good discussion happening around {phrase}.",
                "Worth considering the {phrase} perspective."
            ]
        
        # Select phrase
        if common_phrases:
            phrase = random.choice(common_phrases)
        else:
            phrase = "this topic"
        
        # Generate content
        template = random.choice(templates)
        post = template.format(phrase=phrase)
        
        return post

In [2]:
class SimplifiedCGANMethod:
    """CGAN method"""
    
    def __init__(self, train_df, model_wrapper=None):
        self.train_df = train_df
        self.model_wrapper = model_wrapper
        
        # Simplified CGAN, reduce training epochs and network complexity
        self.cgan_method = CGANMethod(train_df, epochs=15, batch_size=16)  # Reduced from 30 to 15
        
        # Environment analysis cache
        self.env_cache = {}
        
        # User data cache
        self.user_cache = {}
        
        print(f"  CGAN method initialized, training epochs: 15")
    
    def _get_user_posts_cached(self, user_name):
        """Cache user posts to avoid repeated reading"""
        if user_name not in self.user_cache:
            self.user_cache[user_name] = self.train_df[
                self.train_df['userName'] == user_name
            ]['text'].dropna().tolist()
        return self.user_cache[user_name]
    
    def _analyze_environment_cached(self, user_name):
        """Cache environment analysis results"""
        if user_name in self.env_cache:
            return self.env_cache[user_name]
        
        user_posts = self._get_user_posts_cached(user_name)
        
        # Environment analysis
        env_context = {
            'sentiment': 'neutral',
            'common_words': [],
            'topics': []
        }
        
        if user_posts:
            # Only analyze first 5 posts to reduce processing time
            all_text = ' '.join([p.lower() for p in user_posts[:5] if isinstance(p, str)])
            words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text)
            
            # Stopword filtering
            stopwords = {'the', 'and', 'you', 'for', 'are', 'this', 'that'}
            common_words = [w for w, f in Counter(words).most_common(8) if w not in stopwords]
            
            env_context['common_words'] = common_words
        
        self.env_cache[user_name] = env_context
        return env_context
    
    def generate_for_user(self, user_name, num_posts=1):
        try:

            if not self.is_trained:
                print(f"  [CGAN] Starting simplified training...")
                self._train_simple()

            condition = self._encode_condition(user_name)
            

            if self.is_trained and self.cgan is not None:
                generated = self._generate_with_cgan(condition, num_posts)
                if generated and len(generated) > 0 and generated[0].strip():
                    return generated[:num_posts]

            return self._fallback_generation(user_name, num_posts)
            
        except Exception as e:
            print(f"  [CGAN] Generation exception: {e}")
            return self._fallback_generation(user_name, num_posts)

    def _generate_with_cgan(self, condition, num_posts):

        try:
            if self.cgan:
         
                cond_input = condition.reshape(1, -1)
                generated = self.cgan.generate(
                    conditions=cond_input, 
                    num_samples=num_posts, 
                    temperature=0.8
                )
                return generated
        except Exception as e:
            print(f"  CGAN generation failed: {e}")
        return None

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

class TextCGANGenerator(nn.Module):
    """CGAN generator: RNN model for conditional text generation - fixed dimension issues"""
    
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, condition_dim=50, max_length=20):
        super(TextCGANGenerator, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.condition_dim = condition_dim
        self.max_length = max_length
        
        # Fix: ensure condition projection layer output dimension is correct
        self.condition_fc = nn.Linear(condition_dim, hidden_dim)
        
        # Word embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # GRU layer - Fix: input dimension = embedding_dim + hidden_dim
        gru_input_dim = embedding_dim + hidden_dim
        self.gru = nn.GRU(
            gru_input_dim,
            hidden_dim,
            batch_first=True,
            num_layers=1,  # Reduce layers to simplify
            dropout=0.0    # No dropout to simplify
        )
        
        # Output layer
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        for name, param in self.named_parameters():
            if 'weight' in name:
                if len(param.shape) > 1:
                    nn.init.xavier_uniform_(param)
                else:
                    nn.init.normal_(param, 0.0, 0.02)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)
    
    def forward(self, noise, condition, temperature=1.0):
        """
        Forward pass - fixed dimension issues
        
        Args:
            noise: Noise vector [batch_size, noise_dim]
            condition: Condition vector [batch_size, condition_dim]
            temperature: Temperature parameter to control randomness
        """
        batch_size = condition.size(0)
        
        # Process condition
        cond_hidden = torch.tanh(self.condition_fc(condition))  # [batch_size, hidden_dim]
        
        # Initial input (start token = 1)
        input_token = torch.ones(batch_size, 1, dtype=torch.long, device=condition.device)
        
        # Initialize hidden state
        hidden = cond_hidden.unsqueeze(0)  # [1, batch_size, hidden_dim]
        
        # Store outputs
        generated_tokens = []
        
        # Autoregressive generation
        for t in range(self.max_length):
            # Word embedding
            embedded = self.embedding(input_token)  # [batch_size, 1, embedding_dim]
            
            # Combine with condition
            cond_expanded = cond_hidden.unsqueeze(1)  # [batch_size, 1, hidden_dim]
            combined_input = torch.cat([embedded, cond_expanded], dim=2)  # [batch_size, 1, embedding_dim+hidden_dim]
            
            # GRU
            output, hidden = self.gru(combined_input, hidden)
            
            # Generate logits
            logits = self.fc_out(output.squeeze(1)) / temperature
            
            # Sample next token
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            
            # Store result
            generated_tokens.append(next_token)
            
            # Prepare input for next time step
            input_token = next_token
        
        # Combine all time steps
        if generated_tokens:
            tokens = torch.cat(generated_tokens, dim=1)  # [batch_size, seq_len]
        else:
            tokens = torch.zeros(batch_size, self.max_length, dtype=torch.long, device=condition.device)
        
        return tokens


class TextCGANDiscriminator(nn.Module):
    """CGAN discriminator: judge whether text is real and meets conditions - fixed version"""
    
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, condition_dim=50):
        super(TextCGANDiscriminator, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        
        # Word embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Condition projection layer - Fix: ensure input dimension is correct
        self.condition_proj = nn.Linear(condition_dim, hidden_dim)
        
        # BiLSTM encoder
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim // 2,  # Divided by 2 because it's bidirectional
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )
        
        # Discriminator head - Fix: ensure input dimension is correct
        # BiLSTM output dimension: hidden_dim (since bidirectional, hidden_dim//2 * 2 = hidden_dim)
        # Condition vector dimension: condition_dim
        # Combined dimension: hidden_dim + condition_dim
        discriminator_input_dim = hidden_dim + condition_dim
        
        self.discriminator_head = nn.Sequential(
            nn.Linear(discriminator_input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        for name, param in self.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)
    
    def forward(self, tokens, condition):
        """
        Forward pass
        
        Args:
            tokens: Input token sequence [batch_size, seq_len]
            condition: Condition vector [batch_size, condition_dim]
        
        Returns:
            validity: Validity score [batch_size, 1]
        """
        batch_size, seq_len = tokens.size()

        embedded = self.embedding(tokens)  # [batch_size, seq_len, embedding_dim]
        lstm_out, _ = self.lstm(embedded)
        text_features = lstm_out[:, -1, :]  # [batch_size, hidden_dim]
        cond_projected = torch.tanh(self.condition_proj(condition))  # [batch_size, hidden_dim]      
        combined = torch.cat([text_features, condition], dim=1)  # [batch_size, hidden_dim + condition_dim]
        
        validity = self.discriminator_head(combined)
        
        return validity

class TextCGAN:
    """Text CGAN training and generation class"""
    
    def __init__(self, vocab_size, condition_dim=50, device=None, max_length=30):
        self.vocab_size = vocab_size
        self.condition_dim = condition_dim
        self.max_length = max_length
        
        # Device setup
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = device
        
        print(f"CGAN device: {self.device}")
        
        # Initialize generator and discriminator
        self.generator = TextCGANGenerator(
            vocab_size=vocab_size,
            condition_dim=condition_dim,
            max_length=max_length
        ).to(self.device)
        
        self.discriminator = TextCGANDiscriminator(
            vocab_size=vocab_size,
            condition_dim=condition_dim
        ).to(self.device)
        
        # Optimizers
        self.g_optimizer = torch.optim.Adam(self.generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
        self.d_optimizer = torch.optim.Adam(self.discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))
        
        # Loss function
        self.adversarial_loss = nn.BCELoss()
        
        # Vocabulary (token to word mapping)
        self.vocab = {}
        self.idx2word = {}
        
    def build_vocab(self, texts, min_freq=1):
        """Build vocabulary - fixed version"""
        from collections import Counter
        
        # Tokenization
        all_words = []
        for text in texts:
            if isinstance(text, str):
                words = text.lower().split()
                all_words.extend(words)
        
        if not all_words:
            # If no vocabulary, create default vocabulary
            self.vocab = {'<PAD>': 0, '<SOS>': 1, '<UNK>': 2}
            self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<UNK>'}
            print(f"Vocabulary size: {len(self.vocab)} (default)")
            return self.vocab
        
        # Count word frequencies
        word_counts = Counter(all_words)
        
        # Build vocabulary (reserve 0 for padding, 1 for start token, 2 for unknown words)
        self.vocab = {'<PAD>': 0, '<SOS>': 1, '<UNK>': 2}
        
        # Add high frequency words
        idx = 3
        for word, count in word_counts.items():
            if count >= min_freq and idx < self.vocab_size:
                self.vocab[word] = idx
                idx += 1
        
        # Reverse mapping
        self.idx2word = {idx: word for word, idx in self.vocab.items()}
        
        print(f"Vocabulary size: {len(self.vocab)}")
        
        return self.vocab
    def text_to_tensor(self, texts, max_length=None):
        """Convert text to tensor"""
        if max_length is None:
            max_length = self.max_length
        
        batch_size = len(texts)
        tensors = torch.zeros(batch_size, max_length, dtype=torch.long)
        
        for i, text in enumerate(texts):
            if isinstance(text, str):
                words = text.lower().split()[:max_length]
                for j, word in enumerate(words):
                    tensors[i, j] = self.vocab.get(word, 2)  # Unknown words use 2
        
        return tensors.to(self.device)
    
    def tensor_to_text(self, tensors):
        """Convert tensor to text"""
        texts = []
        
        if len(tensors.shape) == 1:
            tensors = tensors.unsqueeze(0)
        
        for i in range(tensors.size(0)):
            tokens = tensors[i].cpu().numpy()
            words = []
            for token in tokens:
                if token == 0:  # PAD token
                    continue
                if token == 1:  # SOS token
                    continue
                word = self.idx2word.get(int(token), '<UNK>')
                if word == '<UNK>':
                    continue
                words.append(word)
            text = ' '.join(words)
            texts.append(text)
        
        return texts
    
    # def train_step(self, real_texts, conditions, labels=None):
    def train_step(self, real_texts, conditions, labels=None):
        "Single training step""
        batch_size = len(real_texts)
        

        real_labels = torch.ones(batch_size, 1, device=self.device) * 0.9
        fake_labels = torch.zeros(batch_size, 1, device=self.device)

        real_tokens = self.text_to_tensor(real_texts)
        conditions_tensor = torch.tensor(conditions, device=self.device, dtype=torch.float32)
        
 
        self.d_optimizer.zero_grad()
        

        real_validity = self.discriminator(real_tokens, conditions_tensor)
        d_real_loss = self.adversarial_loss(real_validity, real_labels)
        

        noise = torch.randn(batch_size, 50, device=self.device)
        fake_tokens = self.generator(noise, conditions_tensor, temperature=0.8) 
        
     
        fake_validity = self.discriminator(fake_tokens.detach(), conditions_tensor)
        d_fake_loss = self.adversarial_loss(fake_validity, fake_labels)
        
        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        self.d_optimizer.step()
        

        self.g_optimizer.zero_grad()
        
      
        fake_tokens = self.generator(noise, conditions_tensor, temperature=0.8)
        fake_validity = self.discriminator(fake_tokens, conditions_tensor)
        g_loss = self.adversarial_loss(fake_validity, real_labels)
        
        g_loss.backward()
        self.g_optimizer.step()
        
        return d_loss.item(), g_loss.item()
    
    def generate(self, conditions, num_samples=1, temperature=0.8):
        """Generate text - fixed version"""
        self.generator.eval()
        
        with torch.no_grad():
            # Prepare conditions
            if isinstance(conditions, list):
                conditions = np.array(conditions)
            
            # Ensure correct dimension
            if len(conditions.shape) == 1:
                conditions = conditions.reshape(1, -1)
            
            conditions_tensor = torch.tensor(conditions, device=self.device, dtype=torch.float32)
            
            # Generate noise
            noise = torch.randn(conditions_tensor.size(0), 50, device=self.device)
            
            # Generate text
            tokens = self.generator(noise, conditions_tensor, temperature=temperature)
            
            # Convert to text
            generated_texts = self.tensor_to_text(tokens)
        
        self.generator.train()
        return generated_texts
        
class CGANMethod:
    """CGAN text generation method - ensures content is always generated"""
    
    def __init__(self, train_df, epochs=5, batch_size=8):
        self.train_df = train_df
        self.epochs = epochs
        self.batch_size = batch_size
        
        # Device
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Prepare training data
        self._prepare_training_data()
        
        # Initialize CGAN
        self.cgan = None
        self.is_trained = False
        
        print(f"  CGAN method initialized, device: {self.device}, training epochs: {epochs}")
    
    def _prepare_training_data(self):
        """Prepare training data"""
        # Extract all texts
        self.all_texts = self.train_df['text'].dropna().tolist()[:100]  # Maximum 100 posts
        
        # Extract user data
        self.user_texts = {}
        for user in self.train_df['userName'].unique()[:5]:  # Maximum 5 users
            user_posts = self.train_df[self.train_df['userName'] == user]['text'].dropna().tolist()[:10]
            if user_posts:
                self.user_texts[user] = user_posts
        
        print(f"  CGAN training data: {len(self.all_texts)} texts, {len(self.user_texts)} users")
    
    def _encode_condition(self, user_name):
        """Encode condition vector"""
        # Simple condition encoding: user ID one-hot + simple features
        users = list(self.user_texts.keys())
        if user_name in users:
            user_idx = users.index(user_name)
        else:
            user_idx = 0
        
        # Create condition vector [user features, random features]
        condition = np.zeros(30)  # condition_dim=30
        
        # User features (10 dimensions)
        if user_idx < 10:
            condition[user_idx] = 1.0
        else:
            condition[user_idx % 10] = 1.0
        
        # Random features (increase diversity)
        condition[10:20] = np.random.randn(10) * 0.1
        
        return condition
    
    def generate_for_user(self, user_name, num_posts=1):
        """Generate content for user - ensures output is always produced"""
        try:
            # If not trained, train first (simplified version)
            if not self.is_trained:
                print(f"  [CGAN] Starting simplified training...")
                self._train_simple()
            
            # Encode condition
            condition = self._encode_condition(user_name)
            
            # Try to generate
            generated = self._generate_with_cgan(condition, num_posts)
            
            if generated and len(generated) > 0:
                result = generated[0]
                # Ensure result is not empty string
                if result and len(result.strip()) > 0:
                    return [result]
            
            # If generation fails, use fallback
            return self._fallback_generation(user_name, num_posts)
            
        except Exception as e:
            print(f"  [CGAN] Generation exception: {e}")
            # Use fallback
            return self._fallback_generation(user_name, num_posts)
    
    # def _train_simple(self):
    def _train_simple(self):
 
        print("  CGAN training: Building vocabulary and model...")

        paired_data = []
        for user, texts in self.user_texts.items():
            cond = self._encode_condition(user)
            for text in texts:
                if isinstance(text, str) and len(text.strip()) > 3:
                    paired_data.append((text, cond))

        if len(paired_data) < 8:
            print(f"  Warning: Only {len(paired_data)} training pairs. CGAN requires more data. Using fallback.")
            self.is_trained = False
            return

        self.cgan = TextCGAN(
            vocab_size=500, 
            condition_dim=30, 
            device=self.device, 
            max_length=15
        )
        self.cgan.build_vocab(self.all_texts, min_freq=1)
        
        print(f"  Starting actual CGAN training for {self.epochs} epochs with {len(paired_data)} samples...")

        for epoch in range(self.epochs):
            random.shuffle(paired_data)
            total_d_loss, total_g_loss = 0.0, 0.0
            num_batches = 0
            
            for i in range(0, len(paired_data), self.batch_size):
                batch = paired_data[i:i+self.batch_size]
                texts = [item[0] for item in batch]
                conds = np.array([item[1] for item in batch])
                
                try:
                    d_loss, g_loss = self.cgan.train_step(texts, conds, labels=None)
                    total_d_loss += d_loss
                    total_g_loss += g_loss
                    num_batches += 1
                except Exception as e:
                    print(f"    Batch training error: {e}")
                    continue
            
            if (epoch + 1) % 5 == 0 and num_batches > 0:
                avg_d = total_d_loss / num_batches
                avg_g = total_g_loss / num_batches
                print(f"  Epoch {epoch+1}/{self.epochs}, D_loss: {avg_d:.4f}, G_loss: {avg_g:.4f}")
        
        self.is_trained = True
        print("  CGAN training complete")
    
    def _generate_with_cgan(self, condition, num_posts):
        """Generate using CGAN"""
        try:
            if self.cgan:
                generated = self.cgan.generate([condition], temperature=0.8, num_samples=1)
                return generated
        except Exception as e:
            print(f"  CGAN generation failed: {e}")
        
        return None
    
    def _fallback_generation(self, user_name, num_posts):
        """Fallback generation"""
        # Randomly select from user history
        user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
        
        if user_posts:
            selected = random.choice(user_posts[:min(3, len(user_posts))])
            return [f"CGAN generated: {selected}"]
        else:
            return [f"CGAN generated content for {user_name}"]

class EnhancedCGANMethod:
    """Enhanced CGAN method: combined with environment awareness"""
    
    def __init__(self, train_df, model_wrapper=None):
        self.train_df = train_df
        self.model_wrapper = model_wrapper
        
        # Simple CGAN
        self.cgan_method = CGANMethod(train_df, epochs=30, batch_size=16)
        
        # Environment analysis cache
        self.env_cache = {}
    
    def _analyze_environment(self, user_name):
        """Analyze user environment"""
        if user_name in self.env_cache:
            return self.env_cache[user_name]
        
        # Get user history
        user_posts = self.train_df[self.train_df['userName'] == user_name]['text'].dropna().tolist()
        
        # Simple environment analysis
        env_context = {
            'sentiment': 'neutral',
            'common_words': [],
            'topics': []
        }
        
        if user_posts:
            # Extract common words
            all_text = ' '.join([p.lower() for p in user_posts[:10] if isinstance(p, str)])
            words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text)
            word_freq = Counter(words)
            stopwords = {'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with'}
            common_words = [w for w, f in word_freq.most_common(10) if w not in stopwords]
            
            env_context['common_words'] = common_words           
          
            positive_words = ['good', 'great', 'excellent', 'positive', 'happy']
            negative_words = ['bad', 'poor', 'negative', 'problem', 'issue']
            
            pos_count = sum(1 for text in user_posts[:5] 
                          if any(word in text.lower() for word in positive_words))
            neg_count = sum(1 for text in user_posts[:5] 
                          if any(word in text.lower() for word in negative_words))
            
            if pos_count > neg_count:
                env_context['sentiment'] = 'positive'
            elif neg_count > pos_count:
                env_context['sentiment'] = 'negative'
        
        self.env_cache[user_name] = env_context
        return env_context
    
    def generate_for_user(self, user_name, num_posts=1):
        """Generate content for user - combined with environment awareness"""
        # Analyze environment
        env_context = self._analyze_environment(user_name)
        
        # Use CGAN to generate
        cgan_posts = self.cgan_method.generate_for_user(user_name, num_posts)
        
        # Post-processing: combine with environment features
        enhanced_posts = []
        for post in cgan_posts:
            enhanced = self._enhance_with_environment(post, env_context)
            enhanced_posts.append(enhanced)
        
        return enhanced_posts
    
    def _enhance_with_environment(self, text, env_context):
        """Enhance text with environment features"""
        if not isinstance(text, str):
            return text        
        words = text.split()     
        
        if len(words) < 5 and env_context['common_words']:      
            add_words = random.sample(env_context['common_words'][:3], 
                                     min(2, len(env_context['common_words'])))
            words.extend(add_words)        
        # Adjust sentiment
        if env_context['sentiment'] == 'positive':
            positive_enhancers = ['good', 'great', 'interesting', 'valuable']
            if random.random() > 0.7:
                words.append(random.choice(positive_enhancers))
        elif env_context['sentiment'] == 'negative' and len(words) > 3:           
            neutral_enhancers = ['however', 'although', 'considering']
            if random.random() > 0.8:
                words.insert(random.randint(0, len(words)//2), random.choice(neutral_enhancers))        
        # Recombine
        enhanced_text = ' '.join(words)        
        # Ensure sentence is complete
        if not enhanced_text.endswith(('.', '!', '?')):
            enhanced_text += '.'        
        return enhanced_text.capitalize()

In [4]:
# ========== Improved Evaluation Metrics ==========

class ImprovedEvaluationMetrics:
    """Improved evaluation metrics class"""
    
    def __init__(self, train_df, test_df):
        self.train_df = train_df
        self.test_df = test_df
        
        # Extract environment features
        self.env_features = self._extract_environment_features()
        
        # Check TextBlob availability
        self.use_textblob = TEXTBLOB_AVAILABLE
        if self.use_textblob:
            print("  Using TextBlob for sentiment analysis")
        else:
            print("  TextBlob unavailable, using dictionary-based sentiment analysis")
    
    def _extract_environment_features(self):
        """Extract environment features - enhanced version: extract more features"""
        # Use more data
        recent_posts = self.train_df['text'].dropna().tolist()
        if len(recent_posts) > 100:
            recent_posts = recent_posts[-100:]  # Use last 100 posts
        
        if not recent_posts:
            return {'trending_words': [], 'common_topics': [], 'avg_sentiment': 0.5}
        
        all_text = ' '.join([str(p) for p in recent_posts])
        
        # Extract words
        words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower())
        word_freq = Counter(words)
        
        # Filter stopwords
        stopwords = {'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with', 'have', 'from',
                    'they', 'what', 'when', 'were', 'been', 'also', 'more', 'some', 'such',
                    'only', 'just', 'like', 'will', 'than', 'other', 'there', 'their', 'them'}
        trending_words = [word for word, freq in word_freq.most_common(30) 
                         if word not in stopwords and freq >= 2]
        
        # Extract topics (based on noun phrases)
        topics = []
        for post in recent_posts[:20]:
            if isinstance(post, str):
                # Extract uppercase word sequences (may indicate proper nouns)
                uppercase_phrases = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b', post)
                topics.extend(uppercase_phrases)
        
        topic_freq = Counter(topics)
        common_topics = [topic for topic, freq in topic_freq.most_common(10) if freq >= 2]
        
        # Calculate average sentiment
        avg_sentiment = 0.5
        if self.use_textblob:
            try:
                sentiments = []
                for post in recent_posts[:20]:
                    if isinstance(post, str) and len(post) > 10:
                        blob = TextBlob(post)
                        # TextBlob polarity range is [-1, 1], map to [0, 1]
                        polarity = (blob.sentiment.polarity + 1) / 2
                        sentiments.append(polarity)
                if sentiments:
                    avg_sentiment = np.mean(sentiments)
            except Exception as e:
                print(f"  Sentiment analysis failed: {e}")
        
        return {
            'trending_words': trending_words[:20],
            'common_topics': common_topics[:5],
            'avg_sentiment': avg_sentiment
        }
    
    def calculate_environment_relevance(self, generated_posts, user_name, method_name):
        """Calculate environment relevance - enhanced: multi-dimensional evaluation"""
        if not generated_posts:
            return 0.4 
        
        gen_text = ' '.join([str(p) for p in generated_posts])
        
        # 1. Vocabulary match
        gen_words = set(re.findall(r'\b[a-zA-Z]{3,}\b', gen_text.lower()))
        env_words = set(self.env_features['trending_words'])
        
        vocab_match = 0.0
        if env_words:
            common_words = gen_words & env_words
            vocab_match = len(common_words) / len(env_words) if env_words else 0
        
        # 2. Topic match
        topic_match = 0.0
        common_topics = self.env_features['common_topics']
        if common_topics:
            topic_count = 0
            for topic in common_topics:
                if topic.lower() in gen_text.lower():
                    topic_count += 1
            topic_match = topic_count / len(common_topics)
        
        # 3. Sentiment match
        sentiment_match = 0.0
        if self.use_textblob:
            try:
                blob = TextBlob(gen_text)
                gen_polarity = (blob.sentiment.polarity + 1) / 2  # Map to [0, 1]
                env_polarity = self.env_features['avg_sentiment']
                sentiment_match = 1 - min(abs(gen_polarity - env_polarity), 0.5) * 2
            except:
                sentiment_match = 0.5
        else:
            # Use vocabulary method to estimate sentiment
            pos_words = ['good', 'great', 'excellent', 'positive', 'optimistic', 'happy', 'progress']
            neg_words = ['bad', 'poor', 'negative', 'problem', 'issue', 'worry', 'concern']
            
            gen_words_list = gen_text.lower().split()
            pos_count = sum(1 for w in gen_words_list if w in pos_words)
            neg_count = sum(1 for w in gen_words_list if w in neg_words)
            
            gen_sentiment = 0.5
            total = pos_count + neg_count
            if total > 0:
                gen_sentiment = pos_count / total
            
            env_polarity = self.env_features['avg_sentiment']
            sentiment_match = 1 - min(abs(gen_sentiment - env_polarity), 0.5) * 2
        
        # Overall score: Vocabulary (40%) + Theme (30%) + Emotion (30%)
        total_score = 0.4 * vocab_match + 0.3 * topic_match + 0.3 * sentiment_match
            
        return min(1.0, max(0.2, total_score))
    
    def calculate_style_similarity(self, generated_posts, ground_truth):
        """Calculate style similarity - comprehensive evaluation version"""
        if not generated_posts or not ground_truth:
            return 0.5
        
        # 1. Length similarity (30%)
        gen_lengths = [len(p.split()) for p in generated_posts if isinstance(p, str)]
        truth_lengths = [len(p.split()) for p in ground_truth[:5] if isinstance(p, str)]
        
        if not gen_lengths or not truth_lengths:
            length_sim = 0.5
        else:
            avg_gen = np.mean(gen_lengths)
            avg_truth = np.mean(truth_lengths)
            if avg_truth > 0:
                length_sim = 1 - min(abs(avg_gen - avg_truth) / avg_truth, 1.0)
            else:
                length_sim = 0.5
        
        # 2. Vocabulary overlap (40%)
        gen_text = ' '.join([str(p).lower() for p in generated_posts])
        truth_text = ' '.join([str(p).lower() for p in ground_truth[:5]])
        
        # Extract words (exclude short words and common words)
        gen_words = set(re.findall(r'\b[a-zA-Z]{3,}\b', gen_text))
        truth_words = set(re.findall(r'\b[a-zA-Z]{3,}\b', truth_text))
        
        common_words = {'the', 'and', 'you', 'for', 'are', 'this', 'that', 'with', 'have', 'from',
                       'they', 'what', 'when', 'were', 'been', 'also', 'more', 'some', 'such',
                       'only', 'just', 'like', 'will', 'than', 'other', 'there', 'their', 'them'}
        
        gen_words_filtered = gen_words - common_words
        truth_words_filtered = truth_words - common_words
        
        if truth_words_filtered:
            word_overlap = len(gen_words_filtered & truth_words_filtered) / len(truth_words_filtered)
        else:
            word_overlap = 0.3  # Default value
        
        # 3. Punctuation usage similarity (15%)
        def get_punctuation_rate(text, punct):
            return text.count(punct) / len(text) if len(text) > 0 else 0
        
        punct_scores = []
        for punct in ['.', ',', '!', '?']:
            gen_rate = get_punctuation_rate(gen_text, punct)
            truth_rate = get_punctuation_rate(truth_text, punct)
            if truth_rate > 0:
                punct_sim = 1 - min(abs(gen_rate - truth_rate) / truth_rate, 1.0)
            else:
                punct_sim = 0.5 if gen_rate == 0 else 0.3
            punct_scores.append(punct_sim)
        
        punct_sim = np.mean(punct_scores) if punct_scores else 0.5
        
        # 4. Capitalization usage similarity (15%)
        def get_uppercase_ratio(text):
            letters = [c for c in text if c.isalpha()]
            return sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0
        
        gen_upper = get_uppercase_ratio(gen_text)
        truth_upper = get_uppercase_ratio(truth_text)
        
        if truth_upper > 0:
            upper_sim = 1 - min(abs(gen_upper - truth_upper) / truth_upper, 1.0)
        else:
            upper_sim = 0.5 if gen_upper == 0 else 0.3
        
        # Combined score
        final_score = (
            0.3 * length_sim + 
            0.4 * min(1.0, word_overlap * 1.5) +  # Amplify vocabulary overlap impact
            0.15 * punct_sim + 
            0.15 * upper_sim
        )
        
        return min(1.0, max(0.1, final_score))
    
    def calculate_diversity(self, generated_posts):
        """Calculate diversity - improved version"""
        if not generated_posts:
            return 0.4
        
        all_words = []
        for post in generated_posts:
            if isinstance(post, str):
                # Extract meaningful words (at least 3 letters)
                words = re.findall(r'\b[a-zA-Z]{3,}\b', post.lower())
                all_words.extend(words)
        
        if len(all_words) < 3:
            return 0.4
        
        unique_words = set(all_words)
        diversity = len(unique_words) / len(all_words)
        
        # Adjust score: if vocabulary is large enough, give bonus
        if len(all_words) >= 10:
            diversity = min(1.0, diversity * 1.2)
        
        return min(1.0, max(0.2, diversity))
    
    def calculate_habit_consistency(self, generated_posts, ground_truth):
        """Calculate habit consistency - improved version: more reasonable score range"""
        if not generated_posts or not ground_truth:
            return 0.5
        
        # Check punctuation usage habits
        def get_punctuation_stats(texts):
            if not texts:
                return {'!': 0.01, '?': 0.01, '.': 0.1, ',': 0.05, 'avg_len': 10}
            
            texts_str = [str(t) for t in texts]
            total_chars = sum(len(t) for t in texts_str)
            
            stats = {}
            for punct in ['!', '?', '.', ',']:
                count = sum(t.count(punct) for t in texts_str)
                rate = count / total_chars if total_chars > 0 else 0.01
                stats[punct] = rate
            
            # Calculate average sentence length
            sentence_lengths = []
            for text in texts_str:
                sentences = re.split(r'[.!?]+', text)
                for sentence in sentences:
                    words = sentence.strip().split()
                    if words:  # Only count non-empty sentences
                        sentence_lengths.append(len(words))
            
            avg_len = np.mean(sentence_lengths) if sentence_lengths else 10
            stats['avg_len'] = avg_len
            
            return stats
        
        gen_stats = get_punctuation_stats(generated_posts)
        truth_stats = get_punctuation_stats(ground_truth[:5])
        
        # Calculate punctuation consistency
        punct_scores = []
        for punct in ['!', '?', '.', ',']:
            gen_rate = gen_stats.get(punct, 0.01)
            truth_rate = truth_stats.get(punct, 0.01)
            
            # If both are very low (close to 0), consider them consistent
            if gen_rate < 0.02 and truth_rate < 0.02:
                punct_scores.append(1.0)
            elif truth_rate > 0:
                # Use relative difference, avoid division by very small numbers
                diff = abs(gen_rate - truth_rate)
                if diff < 0.01:  # Very small difference
                    punct_scores.append(1.0)
                else:
                    similarity = 1 - min(diff / max(truth_rate, 0.02), 1.0)
                    punct_scores.append(similarity)
            else:
                punct_scores.append(0.5)
        
        punct_consistency = np.mean(punct_scores) if punct_scores else 0.5
        
        # Length difference (more lenient penalty)
        gen_len = gen_stats.get('avg_len', 10)
        truth_len = truth_stats.get('avg_len', 10)
        
        len_diff = abs(gen_len - truth_len)
        if len_diff < 3:  # Difference less than 3 words, no penalty
            len_penalty = 0
        else:
            len_penalty = min(1.0, len_diff / 15)  # More lenient penalty
        
        # Final consistency score = punctuation consistency - length penalty
        consistency = punct_consistency - 0.2 * len_penalty  # Reduce weight of length penalty
        
        # Give base score to avoid scores being too low
        consistency = max(0.3, consistency)
        
        return min(1.0, consistency)
    
    def calculate_positivity_score(self, generated_posts):
        """Calculate positivity score - using TextBlob sentiment analysis"""
        if not generated_posts:
            return 0.5
        
        all_text = ' '.join([str(p) for p in generated_posts])
        
        # Use TextBlob for sentiment analysis
        if self.use_textblob:
            try:
                blob = TextBlob(all_text)
                # TextBlob polarity range is [-1, 1], map to [0, 1]
                polarity = blob.sentiment.polarity
                positivity = (polarity + 1) / 2
                
                # Consider subjectivity (confidence)
                subjectivity = blob.sentiment.subjectivity
                
                # Adjust score: if there is clear emotional tendency (high subjectivity), give slight bonus
                if subjectivity > 0.5 and positivity > 0.5:
                    positivity = min(1.0, positivity * 1.1)
                elif subjectivity > 0.5 and positivity < 0.5:
                    positivity = max(0.0, positivity * 0.9)
                
                return min(1.0, max(0.0, positivity))
                
            except Exception as e:
                print(f"  TextBlob sentiment analysis failed: {e}, using dictionary-based method")
                # Fallback to vocabulary method
        
        # Vocabulary method (fallback)
        words = all_text.lower().split()
        
        # Extended positive word list
        positive_words = [
            'good', 'great', 'excellent', 'positive', 'optimistic', 
            'happy', 'progress', 'success', 'improve', 'better',
            'love', 'like', 'support', 'agree', 'encourage', 'wonderful',
            'amazing', 'fantastic', 'brilliant', 'outstanding', 'superb',
            'awesome', 'terrific', 'perfect', 'fabulous', 'marvelous',
            'exceptional', 'commend', 'praise', 'appreciate', 'thank',
            'grateful', 'pleased', 'delighted', 'satisfied', 'content',
            'joy', 'excited', 'enthusiastic', 'hopeful', 'promising'
        ]
        
        # Extended negative word list
        negative_words = [
            'bad', 'poor', 'negative', 'problem', 'issue', 'worry',
            'concern', 'difficult', 'challenge', 'failure', 'hate',
            'dislike', 'against', 'disagree', 'criticize', 'terrible',
            'awful', 'horrible', 'disappoint', 'frustrate', 'angry',
            'upset', 'sad', 'unhappy', 'dissatisfied', 'complain',
            'critique', 'blame', 'fault', 'wrong', 'error', 'mistake',
            'fail', 'weak', 'poorly', 'inadequate', 'insufficient',
            'problematic', 'trouble', 'difficulty', 'struggle'
        ]
        
        pos_count = sum(1 for word in words if word in positive_words)
        neg_count = sum(1 for word in words if word in negative_words)
        
        total = pos_count + neg_count
        if total > 0:
            positivity = pos_count / total
        else:
            # If no sentiment words, check for sentiment phrases
            positive_phrases = ['well done', 'good job', 'nice work', 'keep it up', 'looking good']
            negative_phrases = ['not good', 'bad idea', 'poor quality', 'needs improvement']
            
            pos_phrase_count = sum(1 for phrase in positive_phrases if phrase in all_text.lower())
            neg_phrase_count = sum(1 for phrase in negative_phrases if phrase in all_text.lower())
            
            if pos_phrase_count > neg_phrase_count:
                positivity = 0.7
            elif neg_phrase_count > pos_phrase_count:
                positivity = 0.3
            else:
                positivity = 0.5
        
        # Adjust: if there are positive words and no negative words, increase score
        if pos_count > 0 and neg_count == 0:
            positivity = min(1.0, positivity * 1.2)
        # If there are negative words and no positive words, decrease score
        elif neg_count > 0 and pos_count == 0:
            positivity = max(0.0, positivity * 0.8)
        
        return min(1.0, max(0.0, positivity))
    
    def calculate_realness_score(self, generated_posts, ground_truth):
        """Calculate realness score - improved version"""
        if not generated_posts or not ground_truth:
            return 0.5
        
        # Check multiple dimensions of authenticity
        scores = []
        
        # 1. Length consistency (30%)
        gen_lengths = [len(p.split()) for p in generated_posts if isinstance(p, str)]
        truth_lengths = [len(p.split()) for p in ground_truth[:5] if isinstance(p, str)]
        
        if gen_lengths and truth_lengths:
            avg_gen = np.mean(gen_lengths)
            avg_truth = np.mean(truth_lengths)
            if avg_truth > 0:
                length_sim = 1 - min(abs(avg_gen - avg_truth) / avg_truth, 1.0)
            else:
                length_sim = 0.5
            scores.append(length_sim * 0.3)
        else:
            scores.append(0.15)
        
        # 2. Vocabulary complexity match (30%)
        def get_vocab_complexity(texts):
            if not texts:
                return 0.5
            all_text = ' '.join([str(t) for t in texts])
            words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text.lower())
            if not words:
                return 0.5
            # Long word ratio as complexity indicator
            long_words = [w for w in words if len(w) >= 6]
            return len(long_words) / len(words)
        
        gen_complexity = get_vocab_complexity(generated_posts)
        truth_complexity = get_vocab_complexity(ground_truth[:5])
        
        complexity_sim = 1 - min(abs(gen_complexity - truth_complexity), 0.5) * 2
        scores.append(complexity_sim * 0.3)
        
        # 3. Structure similarity (40%) - check sentence beginning patterns
        def get_start_patterns(texts, n=3):
            patterns = []
            for text in texts[:3]:
                if isinstance(text, str) and text.strip():
                    first_word = text.strip().split()[0].lower() if text.strip().split() else ""
                    patterns.append(first_word)
            return patterns
        
        gen_starts = get_start_patterns(generated_posts)
        truth_starts = get_start_patterns(ground_truth[:3])
        
        if truth_starts and gen_starts:
            # Check if there are common starting words
            common_starts = set(gen_starts) & set(truth_starts)
            if common_starts:
                start_sim = len(common_starts) / max(len(set(truth_starts)), 1)
            else:
                # Check starting word types (e.g., pronouns, verbs, etc.)
                pronoun_starts = {'i', 'you', 'he', 'she', 'we', 'they', 'it'}
                gen_pronoun = any(start in pronoun_starts for start in gen_starts)
                truth_pronoun = any(start in pronoun_starts for start in truth_starts)
                start_sim = 0.7 if gen_pronoun == truth_pronoun else 0.3
        else:
            start_sim = 0.5
        
        scores.append(start_sim * 0.4)
        
        final_score = sum(scores)
        return min(1.0, max(0.2, final_score))